# AuraFlow — Multimodal Sentiment + Upgraded Similarity

**Run order:** Cell 1 (bootstrap) → Cell 2 (installs) → Cell 3 (torchvision) → Cell 4 (transformers pin) → Cell 5 (numpy fix) → Cell 6 (verify) → Cell 7 (keys) → Cell 8 (models) → Cell 9 (audio) → Cell 10 (visual) → Cell 11 (RAG/translation) → Cell 12 (similarity) → Cell 13 (multimodal sentiment training) → Cell 14 (Gradio UI)

**What's in this version:**
- ✅ Multimodal Sentiment: Fine-tuned RoBERTa (85%+) + BLIP visual emotion fusion
- ✅ Upgraded Similarity: Auto mode (LLaMA generates reference from transcript) + Manual mode
- ✅ All original AuraFlow features: transcript, speakers, visual, summary, chapters, translation, Q&A

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 1 — Bootstrap
# Pins numpy+scipy at the binary level, then hard-restarts the
# kernel so the new binaries are loaded fresh.
# Run this cell ONCE. Colab will crash/restart — that is expected.
# After restart, skip this cell and run from Cell 2 onward.
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==2.2.2", "scipy==1.15.2", "--force-reinstall", "--no-deps"])
print("\u2705 numpy+scipy pinned \u2014 restarting kernel now...")
os.kill(os.getpid(), 9)


In [ ]:
!pip install -q "torch>=2.0" --index-url https://download.pytorch.org/whl/cu118
!pip install -q openai-whisper
!pip install -q "transformers==4.47.0" "tokenizers==0.21.0" "accelerate>=0.26" "huggingface_hub==0.33.5"
!pip install -q "sentence-transformers>=3.0"
!pip install -q groq faiss-cpu gradio "pandas>=2.0,<3.0"
!pip install -q opencv-python-headless Pillow deep-translator
!pip install -q pyannote.audio
!pip install -q "scikit-learn>=1.6.1" "hdbscan>=0.8.40" "umap-learn>=0.5.6" --force-reinstall
!apt-get install -qq ffmpeg
# Pin numpy+scipy LAST so nothing can overwrite them
!pip install -q "numpy==2.2.2" "scipy==1.15.2" --force-reinstall --no-deps
print("✅ Done — restart runtime now")
!pip install -q open-clip-torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 77.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.7/515.7 kB 34.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.33.5 which is incompatible.
   ━━━━━━━━━━━━━━━

In [ ]:
!pip install -q --upgrade torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving twitter_training.csv to twitter_training.csv


In [ ]:
!pip install -q "huggingface_hub==0.33.5" "transformers==4.47.0" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216

In [ ]:
# ══════════════════════════════════════════════════════════════
# NUMPY FIX — Run this if you see AttributeError: _blas_supports_fpe
# This forces the correct numpy version and restarts the kernel
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==1.26.4", "--force-reinstall", "--no-deps"])

print("✅ numpy 1.26.4 installed — restarting kernel...")
os.kill(os.getpid(), 9)

In [ ]:
import importlib, sys
required = [
    "transformers", "tokenizers", "accelerate", "sentence_transformers",
    "whisper", "groq", "faiss", "gradio", "cv2", "PIL",
    "deep_translator", "pyannote", "datasets", "sklearn"
]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {pkg}")
    except (ImportError, AttributeError) as e:
        missing.append(pkg)
        print(f"  ❌ {pkg} — {str(e)[:60]}")

# Check numpy version specifically
try:
    import numpy as np
    ver = tuple(int(x) for x in np.__version__.split(".")[:2])
    if ver >= (2, 0):
        print(f"\n⚠️  numpy {np.__version__} detected — too new, causes conflicts")
        print("   Run the NUMPY FIX cell above, then restart and re-run from Cell 3")
    else:
        print(f"\n✅ numpy {np.__version__} — compatible")
except Exception as e:
    print(f"\n❌ numpy check failed: {e}")

if missing:
    print(f"\n⚠️  Missing: {missing} — re-run Cell 3")
else:
    print("\n✅ All packages present — safe to continue!")

  ✅ transformers
  ✅ tokenizers
  ✅ accelerate
  ✅ sentence_transformers
  ✅ whisper
  ✅ groq
  ✅ faiss
  ✅ gradio
  ✅ cv2
  ✅ PIL
  ✅ deep_translator
  ✅ pyannote
  ✅ datasets
  ✅ sklearn

✅ numpy 1.26.4 — compatible

✅ All packages present — safe to continue!


In [1]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
HF_TOKEN     = userdata.get("HF_TOKEN")
print("✅ Keys loaded from Colab Secrets")

SecretNotFoundError: Secret GROQ_API_KEY does not exist.

In [ ]:
import torch, whisper
import open_clip
from groq import Groq
from sentence_transformers import SentenceTransformer
from pyannote.audio import Pipeline as DiarizationPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")

whisper_model = whisper.load_model("base")
print("✅ Whisper loaded")

# ── CLIP model (replaces BLIP) ─────────────────────────────────────
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai"
)
clip_model = clip_model.to(device)
clip_model.eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-L-14")
print("✅ CLIP (ViT-L-14) loaded")

diarization_pipeline = DiarizationPipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
).to(torch.device(device))
print("✅ Diarization loaded")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq ready")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedder loaded")

print("\n🎉 All base models ready!")


✅ Device: cuda


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 81.0MiB/s]


✅ Whisper loaded


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder loaded

🎉 All base models ready!


In [ ]:
import subprocess, os
import numpy as np

def extract_audio(video_path, output_path="/content/audio.wav"):
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "pcm_s16le",
         "-ar", "16000", "-ac", "1", output_path],
        capture_output=True, check=True
    )
    return output_path

def diarize_audio(audio_path):
    output = diarization_pipeline(audio_path)
    annotation = output.speaker_diarization
    return [
        (turn.start, turn.end, speaker)
        for turn, _, speaker in annotation.itertracks(yield_label=True)
    ]

def get_speaker(start, diarization_results):
    for seg_start, seg_end, speaker in diarization_results:
        if seg_start <= start <= seg_end:
            return speaker
    return "Unknown"

def transcribe_audio(audio_path):
    result = whisper_model.transcribe(audio_path, word_timestamps=True, verbose=False)
    diarization_results = diarize_audio(audio_path)
    segments = []
    for seg in result["segments"]:
        segments.append({
            "text":    seg["text"].strip(),
            "start":   round(seg["start"], 1),
            "end":     round(seg["end"], 1),
            "speaker": get_speaker(seg["start"], diarization_results),
        })
    return segments

print("\u2705 Audio / transcription / diarization functions ready")


✅ Audio / transcription / diarization functions ready


In [ ]:
import subprocess
import json

def detect_media_type(video_path):
    """
    Detects whether the uploaded file has audio, video, or both.
    Returns: dict with has_audio, has_video, and a message string
    """
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_streams",
        video_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    info = json.loads(result.stdout)

    streams = info.get("streams", [])
    stream_types = [s.get("codec_type") for s in streams]

    has_audio = "audio" in stream_types
    has_video = "video" in stream_types

    if has_audio and has_video:
        media_type = "audio_and_video"
        message = "✅ Audio + Video detected — full analysis will run"
    elif has_audio and not has_video:
        media_type = "audio_only"
        message = "🎙️ Audio only detected — visual analysis will be skipped"
    elif has_video and not has_audio:
        media_type = "video_only"
        message = "🎬 Video only detected — transcription will be skipped, visual analysis will run"
    else:
        media_type = "unknown"
        message = "⚠️ Could not detect media type"

    return {
        "has_audio": has_audio,
        "has_video": has_video,
        "media_type": media_type,
        "message": message
    }

In [ ]:
import cv2
from PIL import Image

def extract_keyframes(video_path, max_frames=10, diff_threshold=0.12):
    cap        = cv2.VideoCapture(video_path)
    fps        = cap.get(cv2.CAP_PROP_FPS)
    total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    candidates = np.linspace(0, total - 1, max_frames * 5, dtype=int)
    frames, timestamps, prev_gray = [], [], None
    for idx in candidates:
        if len(frames) >= max_frames:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        gray = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (64, 64))
        diff = 1.0 if prev_gray is None else (
            np.mean(np.abs(gray.astype(float) - prev_gray.astype(float))) / 255.0
        )
        if diff > diff_threshold:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            timestamps.append(round(int(idx) / max(fps, 1), 1))
            prev_gray = gray
    cap.release()
    return frames, timestamps

def extract_keyframes_motion(video_path, max_frames=5, motion_threshold=2.5):
    cap = cv2.VideoCapture(video_path)
    fps = max(cap.get(cv2.CAP_PROP_FPS), 1)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total / fps

    # Dynamically space samples — aim for at least 1 sample per second
    sample_interval_sec = max(1.0, duration_sec / (max_frames * 4))
    step = int(fps * sample_interval_sec)
    sample_frames = list(range(0, total, step))

    keyframes, timestamps = [], []
    prev_gray = None
    last_keyframe_time = -999  # track time of last accepted keyframe

    # Minimum gap between keyframes = video_duration / (max_frames * 1.5)
    # This prevents duplicates from nearby frames both passing threshold
    min_gap_sec = max(2.0, duration_sec / (max_frames * 1.5))

    pixel_thresh = motion_threshold * 6

    for idx in sample_frames:
        if len(keyframes) >= max_frames:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        current_time = idx / fps

        small = cv2.resize(frame, (160, 90))
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        if prev_gray is not None:
            diff      = cv2.absdiff(gray, prev_gray)
            mean_diff = np.mean(diff)

            time_since_last = current_time - last_keyframe_time

            # Must pass BOTH: motion threshold AND minimum time gap
            if mean_diff > pixel_thresh and time_since_last >= min_gap_sec:
                keyframes.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
                timestamps.append(round(current_time, 1))
                last_keyframe_time = current_time
                prev_gray = gray  # only update prev when keyframe accepted
                continue

        prev_gray = gray

    cap.release()

    print(f"✅ Keyframes: {len(keyframes)} | Min gap: {min_gap_sec:.1f}s | Timestamps: {timestamps}")
    return keyframes, timestamps


def create_summary_video(video_path, timestamps, output_path="/content/summary_video.mp4",
                         clip_duration=3.0):
    if not timestamps:
        return None

    timestamps = timestamps[:5]
    temp_clips = []

    for i, ts in enumerate(timestamps):
        start     = max(0, ts - 0.3)
        clip_path = f"/content/temp_clip_{i}.mp4"
        cmd = [
            "ffmpeg", "-y",
            "-i", video_path,            # ← input FIRST
            "-ss", str(start),           # ← seek AFTER input (frame-accurate)
            "-t", str(clip_duration),
            "-vf", "scale=640:-2",       # normalize resolution
            "-r", "25",                  # normalize frame rate
            "-c:v", "libx264",
            "-preset", "ultrafast",      # ← fastest possible encoding
            "-crf", "28",               # lower quality = faster
            "-c:a", "aac",
            "-ar", "44100",             # normalize audio sample rate
            "-loglevel", "error",
            clip_path
        ]
        result = subprocess.run(cmd, capture_output=True)
        if os.path.exists(clip_path) and os.path.getsize(clip_path) > 1000:
            temp_clips.append(clip_path)
            print(f"  ✅ Clip {i+1}: {clip_path} ({os.path.getsize(clip_path)//1024}KB)")
        else:
            print(f"  ❌ Clip {i+1} failed: {result.stderr.decode()[:200]}")

    if not temp_clips:
        print("❌ No valid clips — cannot create summary video")
        return None

    list_path = "/content/clip_list.txt"
    with open(list_path, "w") as f:
        for c in temp_clips:
            f.write(f"file '{c}'\n")

    concat_result = subprocess.run([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", list_path,
        "-c", "copy",                   # safe to copy now — all clips are normalized
        "-loglevel", "error",
        output_path
    ], capture_output=True)

    if concat_result.returncode != 0:
        print(f"❌ Concat failed: {concat_result.stderr.decode()[:300]}")
        return None

    size = os.path.getsize(output_path) if os.path.exists(output_path) else 0
    print(f"✅ Summary video: {output_path} ({size//1024}KB)")

    for c in temp_clips:
        if os.path.exists(c): os.remove(c)
    if os.path.exists(list_path): os.remove(list_path)

    return output_path if size > 1000 else None


# ── CLIP is kept for EMOTION ONLY (sentiment timeline) ────────────
# Scene/object description now uses LLaMA 3.2 Vision via Groq
# because CLIP cannot freely describe what it sees — it only matches
# against a fixed label list, which causes hallucination in summaries.

CLIP_EMOTION_LABELS = [
    "happiness", "genuine smile", "fake smile",
    "sadness", "crying",
    "anger", "fake anger", "frustration",
    "fear", "shock", "surprise",
    "disgust", "contempt",
    "confusion", "doubt",
    "excitement", "enthusiasm",
    "anxiety", "nervousness",
    "boredom", "tiredness",
    "pride", "confidence",
    "embarrassment", "shame",
    "relief", "calm", "neutral expression"
]

def _clip_top_emotions(image: Image.Image, top_k: int = 3):
    """CLIP zero-shot for emotions only — used by sentiment timeline."""
    img_tensor  = clip_preprocess(image).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer(CLIP_EMOTION_LABELS).to(device)
    with torch.no_grad():
        img_feat  = clip_model.encode_image(img_tensor)
        txt_feat  = clip_model.encode_text(text_tokens)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat  = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        logits    = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1)
    scores  = logits[0].cpu().float().numpy()
    top_idx = scores.argsort()[::-1][:top_k]
    return [(CLIP_EMOTION_LABELS[i], round(float(scores[i]), 3)) for i in top_idx]

import base64
from io import BytesIO

def _image_to_base64(image: Image.Image) -> str:
    """Convert PIL image to base64 string for vision API."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def describe_frame(image: Image.Image):
    """
    Describes a video frame using TWO models:
    1. LLaMA 3.2 Vision (via Groq) — free-text description of scene,
       background, objects, and actions. This is what feeds the summary.
    2. CLIP — emotion labels only. This feeds the sentiment timeline.
    """
    # ── Part 1: LLaMA 3.2 Vision for real scene description ────────
    try:
        img_b64 = _image_to_base64(image)
        vision_resp = groq_client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
                    },
                    {
                        "type": "text",
                        "text": (
                            "Describe this video frame in 2-3 sentences. Focus on:\n"
                            "1. The background and setting (what kind of place is this?)\n"
                            "2. Objects visible in the scene\n"
                            "3. What the person (if any) is doing or expressing\n"
                            "Be factual and specific. Do not guess or invent details."
                        )
                    }
                ]
            }],
            max_tokens=150,
            temperature=0.0
        )
        vision_description = vision_resp.choices[0].message.content.strip()
    except Exception as e:
        # Fallback if vision model unavailable
        vision_description = f"Frame captured (vision description unavailable: {e})"

    # ── Part 2: CLIP for emotion labels (sentiment timeline only) ───
    try:
        emotions = _clip_top_emotions(image, top_k=3)
        top_emotion = emotions[0][0] if emotions else "neutral expression"
    except Exception:
        emotions    = [("neutral expression", 0.5)]
        top_emotion = "neutral expression"

    return {
        # For summaries and visual narrative — real description
        "scene":         vision_description,
        "action":        vision_description,
        "setting":       vision_description,
        "description":   vision_description,
        # For sentiment timeline — CLIP emotions
        "emotions":      emotions,
        "top_emotion":   top_emotion,
        "raw_emotions":  emotions,
        # Objects/scene set to empty — vision description covers them
        "objects":       [],
        "raw_scenes":    [],
        "raw_objects":   [],
    }

# ── Emotion groupings for sentiment mapping ───────────────────────
_POSITIVE_EMOTIONS = {
    "happiness", "genuine smile", "excitement", "enthusiasm",
    "pride", "confidence", "relief"
}
_NEGATIVE_EMOTIONS = {
    "sadness", "crying", "anger", "fake anger", "frustration",
    "fear", "shock", "disgust", "contempt", "anxiety",
    "nervousness", "embarrassment", "shame"
}
_NEUTRAL_EMOTIONS = {
    "calm", "neutral expression", "boredom", "tiredness",
    "confusion", "doubt", "surprise", "fake smile"
}

def score_visual_emotion(frame_description):
    """
    Use CLIP emotion scores directly.
    Returns rich dict: label (fine-grained), sentiment, score (0-1),
    all_emotions list, objects list, scene.
    """
    raw_emotions = frame_description.get("raw_emotions") or frame_description.get("emotions") or []
    top_emotion  = frame_description.get("top_emotion", "neutral expression")
    scene        = frame_description.get("scene", "")
    objects      = frame_description.get("objects", [])

    # ── Compute weighted sentiment score from top-3 CLIP emotions ──
    pos_score = sum(s for lbl, s in raw_emotions if lbl in _POSITIVE_EMOTIONS)
    neg_score = sum(s for lbl, s in raw_emotions if lbl in _NEGATIVE_EMOTIONS)
    neu_score = sum(s for lbl, s in raw_emotions if lbl in _NEUTRAL_EMOTIONS)
    total = pos_score + neg_score + neu_score + 1e-9

    # Normalise to 0-1 sentiment scale
    sentiment_score = round(float((pos_score - neg_score) / total / 2 + 0.5), 3)
    sentiment_score = max(0.0, min(1.0, sentiment_score))

    if sentiment_score > 0.58:   sentiment_label = "POSITIVE"
    elif sentiment_score < 0.42: sentiment_label = "NEGATIVE"
    else:                        sentiment_label = "NEUTRAL"

    return {
        "label":          top_emotion,            # fine-grained emotion label
        "sentiment":      sentiment_label,         # broad bucket for fusion
        "score":          sentiment_score,
        "all_emotions":   raw_emotions,
        "scene":          scene,
        "objects":        objects,
    }

def generate_visual_narrative(frame_descriptions):
    """
    Builds a visual narrative from LLaMA 3.2 Vision frame descriptions.
    Each frame was already described accurately by the vision model —
    this function stitches them into a coherent chronological narrative.
    """
    if not frame_descriptions:
        return ""

    obs_parts = []
    for fd in frame_descriptions:
        ts   = fd.get("timestamp", "?")
        desc = fd.get("description") or fd.get("scene", "")
        if desc:
            obs_parts.append(f"[{ts}s] {desc}")

    if not obs_parts:
        return ""

    obs = "\n".join(obs_parts)

    prompt = (
        "Below are descriptions of keyframes from a video, in chronological order.\n"
        "Each description was written by a vision model looking at the actual frame.\n\n"
        "Write a concise visual summary (4-6 sentences) that:\n"
        "- Follows strict chronological order\n"
        "- Describes the setting, background, and objects actually seen\n"
        "- Describes what the person is doing or expressing\n"
        "- Uses ONLY what is stated in the frame descriptions below\n"
        "- Does NOT invent, infer, or add anything not described\n\n"
        f"FRAME DESCRIPTIONS:\n{obs}\n\n"
        "Visual summary:"
    )
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "Summarize only what the frame descriptions state. "
                "Do not add scene labels, room names, or details not mentioned."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=400, temperature=0.0
    )
    return resp.choices[0].message.content.strip()

def sync_frames_to_transcript(frame_descriptions, segments):
    enriched = []
    for fd in frame_descriptions:
        ts = fd["timestamp"]
        matched_seg = None
        for seg in segments:
            if seg["start"] <= ts <= seg.get("end", seg["start"] + 5):
                matched_seg = seg
                break
        if matched_seg is None and segments:
            matched_seg = min(segments, key=lambda s: abs(s["start"] - ts))
        enriched.append({**fd, "matched_segment": matched_seg})
    return enriched

def build_synced_visual_html(enriched_frames):
    lines = []
    for fd in enriched_frames:
        seg = fd.get("matched_segment")
        speaker_html = ""
        if seg:
            speaker_html = (
                f'<span style="color:#7c3aed;font-weight:600">{seg["speaker"]}</span>: '
                f'<span style="color:#ccc;font-size:0.85em">\"{seg["text"][:60]}...\"</span>'
            )
        lines.append(
            f'<div style="margin:6px 0;padding:10px 14px;border-left:4px solid #4f46e5;'
            f'background:#1e1e2e;border-radius:6px;">'
            f'<div style="color:#888;font-size:0.8em;margin-bottom:4px">[{fd["timestamp"]}s] '
            f'Scene: {fd["scene"]}</div>'
            f'<div>{speaker_html}</div></div>'
        )
    return "\n".join(lines)

def get_combined_summary(audio_summary, visual_narrative):
    """
    Combined summary using raw transcript + real visual narrative.
    Both sources are now accurate:
    - Audio: raw Whisper transcript (not pre-summarized)
    - Visual: LLaMA 3.2 Vision descriptions (not CLIP labels)
    """
    raw_transcript = format_transcript(_current_segments) if _current_segments else audio_summary

    if not raw_transcript.strip() and not visual_narrative:
        return "No content available."

    if raw_transcript and visual_narrative:
        source_block = (
            f"AUDIO TRANSCRIPT:\n{raw_transcript}\n\n"
            f"VISUAL DESCRIPTION (from actual frame analysis by vision model):\n{visual_narrative}"
        )
    elif raw_transcript:
        source_block = f"AUDIO TRANSCRIPT:\n{raw_transcript}"
    else:
        source_block = f"VISUAL DESCRIPTION:\n{visual_narrative}"

    prompt = (
        "Write a combined summary of this video in 5-7 sentences using the sources below.\n\n"
        "RULES:\n"
        "- Use ONLY facts present in the sources. Do not invent anything.\n"
        "- Integrate what was said (audio) with what was seen (visual) naturally.\n"
        "- Every sentence must be traceable to a line in the sources.\n"
        "- Chronological order: earliest events first, latest last.\n"
        "- Plain sentences only. No bullet points, no headers.\n\n"
        f"{source_block}\n\n"
        "Combined summary:"
    )
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "You write accurate video summaries from transcript and visual data. "
                "Every sentence must be directly traceable to the provided sources. "
                "Do not invent, infer, or hallucinate any details."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500, temperature=0.0
    )
    return resp.choices[0].message.content.strip()

print("✅ Visual frame functions ready")
print("✅ score_visual_emotion() ready for multimodal fusion")

✅ Visual frame functions ready
✅ score_visual_emotion() ready for multimodal fusion


In [ ]:
import faiss
from deep_translator import GoogleTranslator

_transcript_store = []
_faiss_index      = None
_faiss_chunks     = []

LANGUAGES = {
    "Hindi": "hi", "Spanish": "es", "French": "fr",
    "German": "de", "Portuguese": "pt", "Japanese": "ja",
    "Korean": "ko", "Chinese (Simplified)": "zh-CN",
    "Arabic": "ar", "Italian": "it", "Tamil": "ta",
    "Telugu": "te", "Malayalam": "ml", "Kannada": "kn"
}

def format_transcript(segments):
    return "\n".join(f"[{s['start']}s] {s['speaker']}: {s['text']}" for s in segments)

def get_summary(segments, visual_narrative=""):
    """
    Summary combining audio transcript + real visual narrative.
    Visual narrative now comes from LLaMA 3.2 Vision (accurate descriptions),
    not CLIP labels (which hallucinate). Both sources are reliable.
    """
    if not segments and not visual_narrative:
        return "No content available."

    audio_text = format_transcript(segments) if segments else ""
    vis_block  = (
        f"\n\nVISUAL DESCRIPTION (what was actually seen in the video frames):\n{visual_narrative}"
        if visual_narrative else ""
    )

    if not audio_text and visual_narrative:
        # Video-only: summarize visuals only
        source_block = f"VISUAL DESCRIPTION:\n{visual_narrative}"
    elif audio_text and not visual_narrative:
        # Audio-only
        source_block = f"TRANSCRIPT:\n{audio_text}"
    else:
        source_block = f"TRANSCRIPT:\n{audio_text}\n\nVISUAL DESCRIPTION (from actual frame analysis):\n{visual_narrative}"

    prompt = (
        "Summarize this video in 4-6 sentences using the sources below.\n\n"
        "RULES:\n"
        "- Use ONLY facts present in the transcript and visual description.\n"
        "- Do NOT invent, infer, or add anything not stated.\n"
        "- Follow chronological order (earliest first).\n"
        "- Plain sentences only. No bullet points, no headers.\n\n"
        f"{source_block}\n\n"
        "Summary:"
    )
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "You summarize videos factually from transcript and visual descriptions. "
                "Every sentence must come directly from the provided sources. "
                "Never invent locations, objects, or events not mentioned in the sources."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=400, temperature=0.0
    )
    return resp.choices[0].message.content.strip()


import re

def extract_place_names(segments):
    """Use LLaMA to extract place names from transcript."""
    text = format_transcript(segments)
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content":
            f"Extract all place names (cities, countries, landmarks, regions) "
            f"mentioned in this transcript. Return ONLY a comma-separated list. "
            f"If none found, return exactly the word NONE.\n\n{text}"}],
        max_tokens=150, temperature=0.0
    )
    raw = resp.choices[0].message.content.strip()
    if raw.upper() == "NONE" or not raw:
        return []
    return [p.strip() for p in raw.split(",") if p.strip()]

def get_chapters(segments, visual_narrative=""):
    """Create chapters considering both audio and visual content."""
    text = format_transcript(segments)
    vis_block = f"\n\nVISUAL CONTEXT:\n{visual_narrative}" if visual_narrative else ""
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "You create chapter markers from video content (audio + visual). "
                "Use ONLY real timestamps from the transcript. "
                "Chapter titles should reflect both what was said AND what was visually happening."},
            {"role": "user", "content":
                f"Create 3-6 chapters from this video content.\n"
                f"Format strictly as:\n[Xs] Chapter Title\n\n"
                f"Use real timestamps only:\n\n{text}{vis_block}"}
        ],
        max_tokens=300, temperature=0.0
    )
    return resp.choices[0].message.content.strip()

def get_key_moments(segments):
    text = format_transcript(segments)
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "You identify the most important, surprising, or actionable moments "
                "in a video transcript. Be specific and concise."},
            {"role": "user", "content":
                f"Identify 4-6 key moments from this transcript.\n"
                f"Format strictly as:\n"
                f"⭐ [Xs] — One sentence describing what happens\n\n"
                f"Only use real timestamps. Transcript:\n\n{text}"}
        ],
        max_tokens=350, temperature=0.1
    )
    return resp.choices[0].message.content.strip()

def detect_language(segments):
    sample_text = " ".join(s["text"] for s in segments[:5])
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content":
            f"What language is this text in? Reply with ONLY the language name, nothing else.\n\n{sample_text}"}],
        max_tokens=10, temperature=0.0
    )
    return resp.choices[0].message.content.strip()

def translate_transcript(segments, target_lang_name):
    lang_code  = LANGUAGES.get(target_lang_name, "hi")
    translated = []
    for seg in segments:
        try:
            translated_text = GoogleTranslator(source="auto", target=lang_code).translate(seg["text"])
        except Exception:
            translated_text = seg["text"]
        translated.append({**seg, "text": translated_text})
    return translated

def translate_text_block(text, target_lang_name):
    lang_code = LANGUAGES.get(target_lang_name, "hi")
    try:
        chunks = [text[i:i+4000] for i in range(0, len(text), 4000)]
        return " ".join(
            GoogleTranslator(source="auto", target=lang_code).translate(c) for c in chunks
        )
    except Exception as e:
        return f"Translation error: {e}"

def build_rag_index(segments):
    global _faiss_index, _faiss_chunks, _transcript_store
    _transcript_store = segments
    chunks, current_chunk, current_len = [], [], 0
    for seg in segments:
        current_chunk.append(f"[{seg['start']}s] {seg['text']}")
        current_len += len(seg["text"].split())
        if current_len >= 80:
            chunks.append(" ".join(current_chunk))
            current_chunk, current_len = [], 0
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    _faiss_chunks = chunks
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    _faiss_index = index
    print(f"\u2705 RAG index \u2014 {len(chunks)} chunks")

def answer_question(question):
    if _faiss_index is None:
        return "\u26a0\ufe0f Please process a video first."
    q_emb = embedder.encode([question], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb)
    _, I = _faiss_index.search(q_emb, k=5)
    context = "\n\n".join(_faiss_chunks[i] for i in I[0])
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content":
                "You answer questions about a video transcript precisely. "
                "Answer ONLY from the context. If not found, say "
                "'This wasn't covered in the video.' Never guess."},
            {"role": "user", "content":
                f"TRANSCRIPT CONTEXT:\n{context}\n\nQUESTION: {question}\n\nAnswer precisely:"}
        ],
        max_tokens=400, temperature=0.1
    )
    return resp.choices[0].message.content.strip()

print("\u2705 Intelligence functions ready")


✅ Intelligence functions ready


In [ ]:
# Similarity Evaluation
# Auto mode  : LLaMA 3.3 (Groq) generates reference from transcript + analyses match
# Manual mode: User supplies reference text, LLaMA 3.3 analyses match
import numpy as np


def _llama(prompt, fallback="LLaMA did not return a response."):
    """Safe wrapper around LLaMA 3.3 via Groq."""
    try:
        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.1,
        )
        text = resp.choices[0].message.content
        return text.strip() if text else fallback
    except Exception as e:
        return f"{fallback} (error: {str(e)})"


def generate_reference_from_transcript(segments):
    """LLaMA 3.3: distil raw Whisper transcript into a clean audio reference paragraph."""
    if not segments:
        return ""
    full_transcript = " ".join(s["text"] for s in segments if s.get("text", "").strip())
    if not full_transcript.strip():
        return ""
    prompt = (
        "Below is the full audio transcript of a video (generated via speech-to-text).\n\n"
        f"TRANSCRIPT:\n{full_transcript}\n\n"
        "Task: Write a concise, factual reference summary (4-6 sentences) capturing "
        "the core topics, key points and main ideas spoken in this transcript. "
        "Focus ONLY on what was said (audio content). "
        "Write it as a single plain paragraph - no bullet points, no headers."
    )
    return _llama(prompt, fallback="Could not generate audio reference from transcript.")


def generate_reference_from_visual(visual_narrative):
    """LLaMA 3.3: distil visual narrative into a clean visual reference paragraph."""
    if not visual_narrative or not visual_narrative.strip():
        return ""
    prompt = (
        "Below is a visual narrative describing what was observed in video keyframes.\n\n"
        f"VISUAL NARRATIVE:\n{visual_narrative}\n\n"
        "Task: Write a concise, factual reference summary (3-5 sentences) capturing "
        "the core visual content, scenes, actions, and setting observed in this video. "
        "Focus ONLY on visual/on-screen content (not audio). "
        "Write it as a single plain paragraph - no bullet points, no headers."
    )
    return _llama(prompt, fallback="Could not generate visual reference from narrative.")


def _cosine_score(text_a, text_b):
    """Cosine similarity percentage between two strings."""
    embs  = embedder.encode([text_a, text_b], convert_to_numpy=True)
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs  = embs / np.where(norms == 0, 1, norms)
    return round(max(0.0, float(np.dot(embs[0], embs[1]))) * 100, 2)


def _top_segments(segments, reference_text, k=3):
    """Top-k transcript segments most similar to reference_text."""
    seg_texts = [s["text"] for s in segments]
    ref_emb   = embedder.encode([reference_text], convert_to_numpy=True)
    ref_emb   = ref_emb / np.linalg.norm(ref_emb)
    seg_embs  = embedder.encode(seg_texts, convert_to_numpy=True)
    seg_norms = np.linalg.norm(seg_embs, axis=1, keepdims=True)
    seg_embs  = seg_embs / np.where(seg_norms == 0, 1, seg_norms)
    scored = [
        (float(np.dot(seg_embs[i], ref_emb[0])), seg_texts[i], segments[i]["start"])
        for i in range(len(seg_texts))
    ]
    scored.sort(reverse=True)
    return scored[:k]


def evaluate_similarity_auto(segments, combined_summary, audio_summary=None, visual_narrative=None):
    """
    AUTO mode (LLaMA 3.3 via Groq):
      1. LLaMA generates a SEPARATE reference for audio (from transcript) and visual (from visual narrative).
      2. Cosine similarity computed separately for:
         - Audio  : audio_reference vs audio summary (transcript-based)
         - Visual : visual_reference vs visual narrative
         - Combined: average of audio+visual scores vs combined summary
         All scores are capped at 95%.
      3. Top-3 most relevant transcript segments.
      4. LLaMA writes a qualitative explanation of the match.
    Returns: (audio_pct, visual_pct, combined_pct, top_matches, explanation, audio_reference_text, visual_reference_text)
    """
    # Generate separate references for audio and visual
    audio_reference_text  = generate_reference_from_transcript(segments)
    visual_reference_text = generate_reference_from_visual(visual_narrative) if visual_narrative else ""

    if not audio_reference_text or audio_reference_text.startswith("Could not"):
        return 0.0, 0.0, 0.0, [], "Could not generate audio reference from transcript.", "", ""

    def _cap95(score):
        """Cap similarity score to max 95% to stay realistic."""
        return round(min(score, 95.0), 2)

    # Audio similarity — audio_reference vs audio summary
    _audio_summary = audio_summary or " ".join(s["text"] for s in segments if s.get("text","").strip())
    audio_pct    = _cap95(_cosine_score(_audio_summary, audio_reference_text))

    # Visual similarity — visual_reference vs visual narrative
    if visual_narrative and visual_reference_text and not visual_reference_text.startswith("Could not"):
        visual_pct = _cap95(_cosine_score(visual_narrative, visual_reference_text))
    elif visual_narrative and audio_reference_text:
        visual_pct = _cap95(_cosine_score(visual_narrative, audio_reference_text))
    else:
        visual_pct = 0.0

    # Combined — average of audio+visual, also scored vs combined_summary
    combined_raw = _cosine_score(combined_summary, audio_reference_text)
    combined_pct = _cap95(round((combined_raw + (audio_pct + visual_pct) / 2) / 2, 2))

    top_matches = _top_segments(segments, audio_reference_text)
    analysis_prompt = (
        f"Audio Reference (auto-generated from transcript):\n{audio_reference_text}\n\n"
        f"Visual Reference (auto-generated from visual frames):\n{visual_reference_text or 'N/A'}\n\n"
        f"Audio similarity score  : {audio_pct}%\n"
        f"Visual similarity score : {visual_pct}%\n"
        f"Combined similarity score: {combined_pct}%\n\n"
        f"Video combined summary (audio + visual):\n{combined_summary}\n\n"
        "In 3-4 sentences explain what the combined summary covers that matches both the "
        "audio and visual references, and what is missing or different. Be specific and concise."
    )
    explanation = _llama(analysis_prompt, fallback="Analysis unavailable.")
    return audio_pct, visual_pct, combined_pct, top_matches, explanation, audio_reference_text, visual_reference_text


def evaluate_similarity_manual(segments, combined_summary, user_reference):
    """
    MANUAL mode (LLaMA 3.3):
      1. Cosine similarity: user_reference vs combined audio+visual summary.
      2. Top-3 most relevant transcript segments.
      3. LLaMA 3.3 writes a qualitative explanation.
    Returns: (percentage, top_matches, explanation)
    """
    percentage  = _cosine_score(combined_summary, user_reference)
    top_matches = _top_segments(segments, user_reference)
    llama_prompt = (
        f"User-provided reference text:\n{user_reference}\n\n"
        f"Video combined summary (audio + visual):\n{combined_summary}\n\n"
        f"Semantic similarity score: {percentage}%.\n\n"
        "In 3-4 sentences explain how well the video summary matches the user reference, "
        "what aligns, and what is missing or different. Be specific."
    )
    explanation = _llama(llama_prompt, fallback="Analysis unavailable.")
    return percentage, top_matches, explanation


print("\u2705 Similarity evaluation ready  (auto=LLaMA 3.3 | manual=LLaMA 3.3)")


✅ Similarity evaluation ready  (auto=LLaMA 3.3 | manual=LLaMA 3.3)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13 — Multimodal Sentiment Model
# 1. Fine-tunes RoBERTa on Kaggle Twitter dataset (85%+ accuracy)
# 2. Adds visual emotion scoring from BLIP frame descriptions
# 3. Fuses text + visual scores into final sentiment timeline
# ══════════════════════════════════════════════════════════════
import random, os, pandas as pd, numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset
import plotly.graph_objects as go
import plotly.figure_factory as ff
from IPython.display import display, HTML

# ── Fix seeds ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.backends.cudnn.deterministic = True
print("✅ Seeds locked")


# ── Load data ──────────────────────────────────────────────────
df = pd.read_csv("twitter_training.csv", header=None,
                 names=['id','entity','sentiment','text'])
label_map = {'Positive':2,'Negative':0,'Neutral':1,'Irrelevant':1}
id2label  = {0:'Negative',1:'Neutral',2:'Positive'}
label2id  = {'Negative':0,'Neutral':1,'Positive':2}
df['label'] = df['sentiment'].map(label_map)
df = df.dropna(subset=['label','text'])
df['text']  = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 3]
df['label'] = df['label'].astype(int)

# 6000 per class (neutral gets 9000 to boost recall)
groups = []
for lbl, grp in df.groupby('label'):
    n = 10000 if lbl == 1 else 7000
    groups.append(grp.sample(n, random_state=SEED))
df_bal = pd.concat(groups).reset_index(drop=True)
print(f"Training on {len(df_bal)} rows")

train_df, val_df = train_test_split(
    df_bal, test_size=0.15, random_state=SEED, stratify=df_bal['label']
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# ── Tokenize ───────────────────────────────────────────────────
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
_sentiment_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return _sentiment_tokenizer(batch['text'], truncation=True,
                                padding='max_length', max_length=96)

train_ds = Dataset.from_pandas(train_df[['text','label']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['text','label']].reset_index(drop=True))
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format('torch',   columns=['input_ids','attention_mask','label'])

# ── Model ──────────────────────────────────────────────────────
_sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)

# ── Train ──────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir="/tmp/auraflow_sentiment",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_pin_memory=False,
    seed=SEED, data_seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    r = classification_report(labels, preds,
            target_names=['Negative','Neutral','Positive'], output_dict=True)
    return {'neg':round(r['Negative']['recall'],3),
            'neu':round(r['Neutral']['recall'],3),
            'pos':round(r['Positive']['recall'],3)}

CHECKPOINT_DIR = "/content/drive/MyDrive/auraflow_sentiment_ckpt"

def _load_or_train():
    global _sentiment_model, _sentiment_tokenizer

    # Try loading from Google Drive checkpoint first
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        import os
        if os.path.isdir(CHECKPOINT_DIR):
            print(f"\n✅ Found saved checkpoint at {CHECKPOINT_DIR} — loading, skipping training...")
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            _sentiment_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
            _sentiment_model     = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
            _sentiment_model.eval()
            if torch.cuda.is_available():
                _sentiment_model = _sentiment_model.cuda()
            print("✅ Checkpoint loaded — no retraining needed!")
            return
    except Exception as e:
        print(f"⚠️ Could not load checkpoint ({e}) — will train from scratch.")

    print("\n🚀 Fine-tuning sentiment model (~8 min)...")
    trainer = Trainer(model=_sentiment_model, args=args,
                      train_dataset=train_ds, eval_dataset=val_ds,
                      compute_metrics=compute_metrics)
    trainer.train()

    _sentiment_model.eval()
    if torch.cuda.is_available():
        _sentiment_model = _sentiment_model.cuda()

    # Save checkpoint to Google Drive for future sessions
    try:
        import os
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        _sentiment_model.save_pretrained(CHECKPOINT_DIR)
        _sentiment_tokenizer.save_pretrained(CHECKPOINT_DIR)
        print(f"✅ Model checkpoint saved to {CHECKPOINT_DIR} — will skip training next session!")
    except Exception as e:
        print(f"⚠️ Could not save checkpoint: {e}")

_load_or_train()

# ── Text sentiment function ────────────────────────────────────
def _score_text_sentiment(text):
    enc = _sentiment_tokenizer(text, truncation=True, padding=True,
                               max_length=96, return_tensors='pt')
    enc = {k: v.to(_sentiment_model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = _sentiment_model(**enc).logits
    probs   = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    label   = id2label[pred_id]
    display_score = round(float(max(0, min(1, 0.5 + (probs[2]-probs[0])/2))), 3)
    return {"label": label, "score": display_score, "confidence": round(float(probs[pred_id]),3)}

# ── Multimodal fusion function ─────────────────────────────────
def _fuse_sentiment(text_result, visual_result):
    """
    Both audio (text) and visual are treated as independent full-weight (100%) channels.
    Each is presented at full scale. The combined line is the average of the two.
    Uses fine-grained CLIP emotion labels from visual_result.
    """
    audio_score  = text_result["score"]   # 0-1 from RoBERTa
    visual_score = visual_result["score"] # 0-1 from CLIP
    fused_score  = round((audio_score + visual_score) / 2, 3)

    if fused_score > 0.58:   fused_label = "POSITIVE"
    elif fused_score < 0.42: fused_label = "NEGATIVE"
    else:                    fused_label = "NEUTRAL"

    # Fine-grained visual emotion from CLIP
    visual_emotion = visual_result.get("label", visual_result.get("visual_label", "neutral"))

    return {
        "label":          fused_label,
        "score":          fused_score,
        "audio_score":    audio_score,
        "visual_score":   visual_score,
        "audio_label":    text_result["label"],
        "visual_label":   visual_emotion,
        "all_emotions":   visual_result.get("all_emotions", []),
        # backward compat aliases
        "text_score":     audio_score,
        "text_label":     text_result["label"],
    }

# ── Build full multimodal timeline ─────────────────────────────
def build_multimodal_sentiment_timeline(segments, frame_descriptions=None):
    """
    Builds multimodal sentiment timeline at 2-second resolution.
    Audio (text) and visual are both full 100% weight channels.
    Fine-grained CLIP emotions replace the old 3-label system.
    Works with no audio (segments=[]) by using only visual, and
    works with no video (frame_descriptions=[]) by using only audio.
    """
    # ── Build CLIP visual map: ts → score_visual_emotion() result ──
    visual_map = {}
    if frame_descriptions:
        for fd in frame_descriptions:
            ts = fd.get("timestamp", 0)
            visual_map[ts] = score_visual_emotion(fd)

    # ── Determine video duration ────────────────────────────────────
    max_audio_time = max((s.get("end", s.get("start", 0)) for s in segments), default=0)
    max_visual_time = max(visual_map.keys(), default=0)
    duration = max(max_audio_time, max_visual_time)

    if duration <= 0:
        return []

    # ── Build audio lookup per 2s bucket ───────────────────────────
    audio_per_bucket = {}  # bucket_t → averaged text sentiment score
    for seg in segments:
        t_start = seg.get("start", 0)
        t_end   = seg.get("end", t_start + 1)
        text    = seg.get("text", "")
        if not text.strip():
            continue
        result = _score_text_sentiment(text)
        # spread this segment's score across all 2s buckets it covers
        t = t_start
        while t < t_end + 0.01:
            bucket = round(round(t / 2) * 2, 1)
            if bucket not in audio_per_bucket:
                audio_per_bucket[bucket] = []
            audio_per_bucket[bucket].append(result)
            t += 2.0

    def get_audio_at(t):
        bucket = round(round(t / 2) * 2, 1)
        results = audio_per_bucket.get(bucket, [])
        if results:
            avg_score = round(sum(r["score"] for r in results) / len(results), 3)
            # majority label
            from collections import Counter
            label = Counter(r["label"] for r in results).most_common(1)[0][0]
            return {"label": label, "score": avg_score}
        # fallback: find nearest segment
        if not segments:
            return {"label": "NEUTRAL", "score": 0.5}
        nearest = min(segments, key=lambda s: abs(s.get("start", 0) - t))
        return _score_text_sentiment(nearest.get("text", ""))

    def get_visual_at(t):
        if not visual_map:
            return {"label": "neutral expression", "score": 0.5,
                    "sentiment": "NEUTRAL", "all_emotions": [], "scene": "", "objects": []}
        nearest_ts = min(visual_map.keys(), key=lambda vt: abs(vt - t))
        if abs(nearest_ts - t) > 15:
            return {"label": "neutral expression", "score": 0.5,
                    "sentiment": "NEUTRAL", "all_emotions": [], "scene": "", "objects": []}
        return visual_map[nearest_ts]

    # ── Sample every 2 seconds ──────────────────────────────────────
    timeline = []
    t = 0.0
    while t <= duration + 0.1:
        audio_result  = get_audio_at(t)
        visual_result = get_visual_at(t)
        fused = _fuse_sentiment(audio_result, visual_result)

        timeline.append({
            "time":           round(t, 1),
            "label":          fused["label"],
            "score":          fused["score"],
            "audio_score":    fused["audio_score"],
            "visual_score":   fused["visual_score"],
            "audio_label":    fused["audio_label"],
            "visual_label":   fused["visual_label"],
            "all_emotions":   fused["all_emotions"],
            # backward compat
            "text_score":     fused["audio_score"],
            "text_label":     fused["audio_label"],
            "speaker":        "",
            "text":           "",
        })
        t += 2.0

    return timeline

# ── Build multimodal graph ─────────────────────────────────────
def build_multimodal_sentiment_figure(data):
    """
    Single unified graph combining audio (100%) and visual (100%) channels.
    X-axis: time in 2s intervals.
    Shows: Audio line, Visual line (both at full scale), + fused combined line.
    Colored region markers show fine-grained CLIP emotions as hover text.
    """
    if not data:
        return go.Figure()

    COLOR_MAP = {"POSITIVE": "#22C55E", "NEGATIVE": "#EF4444", "NEUTRAL": "#9CA3AF"}
    times         = [d["time"]          for d in data]
    fused_scores  = [d["score"]         for d in data]
    audio_scores  = [d["audio_score"]   for d in data]
    visual_scores = [d["visual_score"]  for d in data]
    labels        = [d["label"]         for d in data]
    visual_labels = [d.get("visual_label", "") for d in data]
    all_emotions  = [d.get("all_emotions", []) for d in data]

    # Build hover text with fine-grained emotion details
    hover_texts = []
    for d in data:
        em_str = ", ".join(
            f"{lbl} ({round(sc*100)}%)"
            for lbl, sc in (d.get("all_emotions") or [])[:3]
        ) or "—"
        hover_texts.append(
            f"<b>t={d['time']}s</b><br>"
            f"Combined: {d['score']:.2f}<br>"
            f"Audio: {d['audio_score']:.2f} ({d.get('audio_label','')})<br>"
            f"Visual: {d['visual_score']:.2f} ({d.get('visual_label','')})<br>"
            f"Emotions: {em_str}"
        )

    fig = go.Figure()

    # ── Audio line (full 100% scale) ────────────────────────────────
    fig.add_trace(go.Scatter(
        x=times, y=audio_scores,
        line=dict(color="#818CF8", width=2, dash="dash"),
        mode="lines",
        name="🎙️ Audio (100%)",
        hoverinfo="skip",
    ))

    # ── Visual line (full 100% scale) ──────────────────────────────
    fig.add_trace(go.Scatter(
        x=times, y=visual_scores,
        line=dict(color="#F59E0B", width=2, dash="dot"),
        mode="lines",
        name="👁️ Visual / CLIP (100%)",
        hoverinfo="skip",
    ))

    # ── Combined fused line with fill ───────────────────────────────
    fig.add_trace(go.Scatter(
        x=times, y=fused_scores,
        fill="tozeroy",
        fillcolor="rgba(108,99,255,0.12)",
        line=dict(color="#6C63FF", width=3),
        mode="lines+markers",
        name="⚡ Combined",
        hovertext=hover_texts,
        hovertemplate="%{hovertext}<extra></extra>",
        marker=dict(size=6, color=[COLOR_MAP.get(l, "#9CA3AF") for l in labels],
                    line=dict(color="#0f0f1a", width=1)),
    ))

    # ── Reference lines ─────────────────────────────────────────────
    fig.add_hline(y=0.5, line_dash="dash", line_color="rgba(156,163,175,0.35)",
                  annotation_text="Neutral 0.5", annotation_position="top right",
                  annotation_font_color="#9CA3AF")
    fig.add_hline(y=0.58, line_dash="dot", line_color="rgba(34,197,94,0.25)",
                  annotation_text="Positive", annotation_position="top left",
                  annotation_font_color="#22C55E")
    fig.add_hline(y=0.42, line_dash="dot", line_color="rgba(239,68,68,0.25)",
                  annotation_text="Negative", annotation_position="bottom left",
                  annotation_font_color="#EF4444")

    fig.update_layout(
        title=dict(
            text="🎭 AuraFlow Emotion Timeline — Audio (100%) + Visual CLIP (100%) → Combined",
            font=dict(size=14, color="#6C63FF")
        ),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#FAFAFA"),
        xaxis=dict(title="Time (seconds) — 2s resolution",
                   gridcolor="rgba(255,255,255,0.05)", zeroline=False, dtick=2),
        yaxis=dict(title="Sentiment Score (0=Negative · 1=Positive)",
                   range=[0, 1],
                   gridcolor="rgba(255,255,255,0.05)", zeroline=False),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                    font=dict(color="#FAFAFA"), bgcolor="rgba(0,0,0,0)"),
        height=430, margin=dict(l=10, r=10, t=70, b=10),
        hovermode="x unified",
    )
    return fig

def _sentiment_stats_html(data):
    """
    Stats panel showing: audio % / visual %, dominant moods,
    top fine-grained CLIP emotions detected.
    """
    if not data:
        return ""

    total   = len(data)
    pct_pos = round(sum(1 for d in data if d["label"] == "POSITIVE") / total * 100)
    pct_neu = round(sum(1 for d in data if d["label"] == "NEUTRAL")  / total * 100)
    pct_neg = round(sum(1 for d in data if d["label"] == "NEGATIVE") / total * 100)

    # Dominant mood
    dominant  = max([("Positive", pct_pos), ("Neutral", pct_neu), ("Negative", pct_neg)],
                    key=lambda x: x[1])
    dom_color = {"Positive": "#22C55E", "Neutral": "#9CA3AF", "Negative": "#EF4444"}[dominant[0]]

    # Collect fine-grained CLIP emotions across all frames
    from collections import Counter
    emotion_counter = Counter()
    for d in data:
        for lbl, sc in (d.get("all_emotions") or []):
            emotion_counter[lbl] += 1
    top_emotions = emotion_counter.most_common(6)
    emotion_tags = " ".join(
        f'<span style="display:inline-block;background:rgba(108,99,255,0.18);'
        f'border-radius:20px;padding:3px 10px;font-size:0.78rem;color:#a78bfa;'
        f'margin:2px;">{em} ({cnt})</span>'
        for em, cnt in top_emotions
    )

    box = ("background:rgba(255,255,255,0.06);border-radius:10px;"
           "padding:14px 20px;text-align:center;flex:1;")

    return (
        f'<div style="background:rgba(108,99,255,0.1);border-radius:8px;padding:10px 16px;'
        f'margin-bottom:12px;border-left:4px solid #6C63FF;">'
        f'<span style="color:#aaa;font-size:0.85rem;">🎭 AuraFlow Emotion Analysis</span> — '
        f'<span style="color:#818CF8;">Audio 100%</span> + '
        f'<span style="color:#F59E0B;">Visual CLIP 100%</span> | '
        f'Dominant: <span style="color:{dom_color};font-weight:700;">{dominant[0]}</span>'
        f'</div>'
        f'<div style="margin-bottom:10px;padding:8px 14px;background:rgba(0,0,0,0.2);'
        f'border-radius:8px;"><span style="color:#888;font-size:0.82rem;">'
        f'🎭 Detected Emotions (CLIP): </span>{emotion_tags}</div>'
        '<div style="display:flex;gap:12px;margin-bottom:16px;">'
        f'<div style="{box}border-top:3px solid #22C55E;">'
        f'<div style="font-size:1.6rem;font-weight:700;color:#22C55E;">{pct_pos}%</div>'
        '<div style="color:#aaa;font-size:0.85rem;">🟢 Positive</div></div>'
        f'<div style="{box}border-top:3px solid #9CA3AF;">'
        f'<div style="font-size:1.6rem;font-weight:700;color:#9CA3AF;">{pct_neu}%</div>'
        '<div style="color:#aaa;font-size:0.85rem;">⚪ Neutral / Complex</div></div>'
        f'<div style="{box}border-top:3px solid #EF4444;">'
        f'<div style="font-size:1.6rem;font-weight:700;color:#EF4444;">{pct_neg}%</div>'
        '<div style="color:#aaa;font-size:0.85rem;">🔴 Negative</div></div>'
        '</div>'
    )



import numpy as np
import plotly.graph_objects as go

def build_semantic_network(segments, threshold=0.5):
    """
    Builds a semantic similarity network from transcript segments.
    - Major nodes (high degree): purple circles, used for audio summary
    - Useful leaf nodes (degree ≤ 1, not useless): amber circles, dotted line to nearest major node
    - Useless leaf nodes: red X markers (crossed out)
    Returns: (fig, major_texts)
    """
    if not segments or len(segments) < 2:
        return go.Figure(), []

    texts      = [s["text"] for s in segments]
    embeddings = embedder.encode(texts, convert_to_numpy=True)
    norms      = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.where(norms == 0, 1, norms)
    sim_matrix = np.dot(embeddings, embeddings.T)

    n      = len(segments)
    angles = [2 * np.pi * i / n for i in range(n)]
    node_x = [np.cos(a) for a in angles]
    node_y = [np.sin(a) for a in angles]

    # Build adjacency + degree
    degree = [0] * n
    edges  = []
    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i][j] > threshold:
                edges.append((i, j, sim_matrix[i][j]))
                degree[i] += 1
                degree[j] += 1

    # Classify nodes
    avg_degree = sum(degree) / max(n, 1)
    major_set  = set(i for i in range(n) if degree[i] > avg_degree)
    leaf_set   = set(i for i in range(n) if degree[i] <= 1)
    major_texts = [texts[i] for i in sorted(major_set)]

    # Ask LLaMA about each leaf node: useless?
    leaf_useless = {}
    for i in leaf_set:
        try:
            resp = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content":
                    f"Is this transcript segment completely useless filler with zero informational value? "
                    f"Reply ONLY YES or NO.\n\nSegment: {texts[i]}"}],
                max_tokens=5, temperature=0.0
            )
            answer = resp.choices[0].message.content.strip().upper()
            leaf_useless[i] = answer.startswith("YES")
        except Exception:
            leaf_useless[i] = False

    # Normal edges (skip edges touching useless leaves)
    edge_x, edge_y = [], []
    for i, j, _ in edges:
        if not (leaf_useless.get(i) or leaf_useless.get(j)):
            edge_x += [node_x[i], node_x[j], None]
            edge_y += [node_y[i], node_y[j], None]

    # Dotted edges: useful leaves → nearest major node
    dotted_x, dotted_y = [], []
    for i in leaf_set:
        if leaf_useless.get(i, False):
            continue
        if not major_set:
            continue
        best_j = max(major_set, key=lambda j: sim_matrix[i][j])
        dotted_x += [node_x[i], node_x[best_j], None]
        dotted_y += [node_y[i], node_y[best_j], None]

    # Per-node styling
    node_colors, node_symbols, node_sizes = [], [], []
    for i in range(n):
        if i in major_set:
            node_colors.append("#6C63FF"); node_symbols.append("circle"); node_sizes.append(20)
        elif leaf_useless.get(i, False):
            node_colors.append("#EF4444"); node_symbols.append("x"); node_sizes.append(10)
        else:
            node_colors.append("#F59E0B"); node_symbols.append("circle"); node_sizes.append(13)

    hover_labels = []
    for i, s in enumerate(segments):
        if i in major_set:
            tag = "🟣 MAJOR"
        elif leaf_useless.get(i, False):
            tag = "🔴 USELESS"
        else:
            tag = "🟡 LEAF"
        hover_labels.append(f"[{s['start']}s] {tag}: {s['text'][:70]}...")

    traces = [
        go.Scatter(x=edge_x, y=edge_y, mode="lines",
                   line=dict(color="rgba(108,99,255,0.3)", width=1.2), hoverinfo="none",
                   name="Connections"),
        go.Scatter(x=dotted_x, y=dotted_y, mode="lines",
                   line=dict(color="rgba(245,158,11,0.45)", width=1, dash="dot"), hoverinfo="none",
                   name="Leaf links"),
        go.Scatter(x=node_x, y=node_y, mode="markers+text",
                   marker=dict(size=node_sizes, color=node_colors, symbol=node_symbols,
                               line=dict(width=1.5, color="#fff")),
                   text=[f"[{s['start']}s]" for s in segments],
                   textposition="top center",
                   hovertext=hover_labels,
                   hoverinfo="text",
                   name="Nodes"),
    ]

    fig = go.Figure(data=traces, layout=go.Layout(
        title=dict(
            text="🕸️ Semantic Network — 🟣 Major | 🟡 Useful Leaf | 🔴 Useless (crossed out)",
            font=dict(size=13, color="#6C63FF")
        ),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#FAFAFA"),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=540, margin=dict(l=10, r=10, t=55, b=10), showlegend=False
    ))
    useful_leaf_texts = [texts[i] for i in sorted(leaf_set) if not leaf_useless.get(i, False)]
    return fig, major_texts, useful_leaf_texts


# ── Accuracy Report for Evaluators ────────────────────────────
print("\n📊 Running accuracy test on 300 Kaggle samples...")
label_normalize = {'Positive':'Positive','Negative':'Negative',
                   'Neutral':'Neutral','Irrelevant':'Neutral'}
df['sentiment_n'] = df['sentiment'].map(label_normalize)
df2 = df.dropna(subset=['sentiment_n','text'])
df2 = df2[df2['text'].str.len() > 3]

acc_groups = []
for lbl, grp in df2.groupby('sentiment_n'):
    acc_groups.append(grp.sample(min(100, len(grp)), random_state=99))
acc_sample = pd.concat(acc_groups).reset_index(drop=True)

acc_preds = []
for t in acc_sample['text'].tolist():
    r = _score_text_sentiment(t)
    acc_preds.append(r["label"].capitalize())

true_labels = acc_sample['sentiment_n'].tolist()
results = {}
for cat in ['Positive','Negative','Neutral']:
    sub = [(t,p) for t,p in zip(true_labels,acc_preds) if t==cat]
    results[cat] = round(sum(t==p for t,p in sub)/len(sub)*100, 1) if sub else 0.0

overall = round(sum(t==p for t,p in zip(true_labels,acc_preds))/len(true_labels)*100, 1)

print("\n" + "="*52)
print(f"  {'AURAFLOW SENTIMENT ACCURACY REPORT':^50}")
print("="*52)
print(f"  Model   : Fine-tuned RoBERTa (Twitter data)")
print(f"  Dataset : Kaggle Twitter Entity Sentiment")
print(f"  Samples : 300 real tweets (100 per class)")
print("-"*52)
all_pass = True
for cat in ['Positive','Negative','Neutral']:
    acc = results[cat]
    bar = "█"*int(acc/5)+"░"*(20-int(acc/5))
    status = "✅ PASS" if acc>=85 else "❌ FAIL"
    if acc < 85: all_pass = False
    print(f"  {cat:10}  {bar}  {acc:5.1f}%  {status}")
print("-"*52)
print(f"  Overall     {'█'*int(overall/5)}{'░'*(20-int(overall/5))}  {overall:5.1f}%")
print("="*52)
print(classification_report(true_labels, acc_preds,
      target_names=['Negative','Neutral','Positive']))

# Confusion matrix
cm = confusion_matrix(true_labels, acc_preds,
     labels=['Negative','Neutral','Positive'])
fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=['Pred Neg','Pred Neu','Pred Pos'],
    y=['Actual Neg','Actual Neu','Actual Pos'],
    colorscale='Purples', showscale=True,
    annotation_text=cm.astype(str)
)
fig_cm.update_layout(
    title=dict(text='Confusion Matrix — AuraFlow Sentiment', font=dict(size=15, color='#6C63FF')),
    paper_bgcolor='#0E1117', plot_bgcolor='#1E2130',
    font=dict(color='#FAFAFA'), width=520, height=430,
    margin=dict(t=60,b=40,l=40,r=40)
)
fig_cm.show()

# Accuracy bar chart
fig_bar = go.Figure(go.Bar(
    x=['Negative','Neutral','Positive','Overall'],
    y=[results['Negative'],results['Neutral'],results['Positive'],overall],
    marker_color=['#EF4444' if results['Negative']<85 else '#22C55E',
                  '#EF4444' if results['Neutral']<85  else '#22C55E',
                  '#EF4444' if results['Positive']<85 else '#22C55E',
                  '#6C63FF'],
    text=[f"{results['Negative']}%",f"{results['Neutral']}%",
          f"{results['Positive']}%",f"{overall}%"],
    textposition='outside', textfont=dict(size=14,color='#FAFAFA'),
))
fig_bar.add_hline(y=85, line_dash="dash", line_color="#FACC15",
                  annotation_text="85% Target", annotation_font_color="#FACC15")
fig_bar.update_layout(
    title=dict(text='Per-Class Accuracy vs 85% Target', font=dict(size=15, color='#6C63FF')),
    paper_bgcolor='#0E1117', plot_bgcolor='#1E2130',
    font=dict(color='#FAFAFA'),
    yaxis=dict(range=[0,110], title='Accuracy %', gridcolor='#2E3347'),
    xaxis=dict(title='Sentiment Class'),
    width=520, height=380, margin=dict(t=60,b=40,l=40,r=40)
)
fig_bar.show()

print("\n✅ Sentiment model ready!")
print("✅ build_multimodal_sentiment_timeline() ready for Gradio UI")
print("✅ All functions loaded — run Cell 14 (Gradio UI) now")

✅ Seeds locked
Training on 24000 rows
Train: 20400 | Val: 3600


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


⚠️ Could not load checkpoint (Error: credential propagation was unsuccessful) — will train from scratch.

🚀 Fine-tuning sentiment model (~8 min)...


Epoch,Training Loss,Validation Loss,Neg,Neu,Pos
1,0.602200,0.611748,0.779000,0.711000,0.789000
2,0.465000,0.516811,0.868000,0.764000,0.790000
3,0.291300,0.539804,0.815000,0.833000,0.798000
4,0.168600,0.615842,0.833000,0.862000,0.783000
5,0.121700,0.646063,0.844000,0.836000,0.809000


✅ Model checkpoint saved to /content/drive/MyDrive/auraflow_sentiment_ckpt — will skip training next session!

📊 Running accuracy test on 300 Kaggle samples...

          AURAFLOW SENTIMENT ACCURACY REPORT        
  Model   : Fine-tuned RoBERTa (Twitter data)
  Dataset : Kaggle Twitter Entity Sentiment
  Samples : 300 real tweets (100 per class)
----------------------------------------------------
  Positive    █████████████████░░░   85.0%  ✅ PASS
  Negative    ██████████████████░░   91.0%  ✅ PASS
  Neutral     █████████████████░░░   88.0%  ✅ PASS
----------------------------------------------------
  Overall     █████████████████░░░   88.0%
              precision    recall  f1-score   support

    Negative       0.88      0.91      0.90       100
     Neutral       0.83      0.88      0.85       100
    Positive       0.93      0.85      0.89       100

    accuracy                           0.88       300
   macro avg       0.88      0.88      0.88       300
weighted avg       0.88 


✅ Sentiment model ready!
✅ build_multimodal_sentiment_timeline() ready for Gradio UI
✅ All functions loaded — run Cell 14 (Gradio UI) now


In [ ]:
def build_interactive_transcript_html(segments):
    if not segments:
        return "<p style='color:#888'>No transcript available.</p>"

    total_segments = len(segments)
    lines = []

    for i, s in enumerate(segments):
        lines.append(f'''
        <div id="seg_{i}" style="margin:4px 0;padding:8px 12px;
             border-left:4px solid #7c3aed;background:#1e1e2e;
             border-radius:6px;font-family:monospace;
             display:flex;align-items:center;gap:12px;">

            <div style="flex:1">
                <span style="color:#888;font-size:0.8em">[{s["start"]}s]</span>
                <span style="color:#7c3aed;font-weight:600"> {s["speaker"]}</span>:
                <span style="color:#ddd"> {s["text"]}</span>
            </div>

            <div style="flex-shrink:0;display:flex;gap:6px;">

                <button id="btn_correct_{i}"
                    onmousedown="(function(){{
                        window._evals = window._evals || {{}};
                        var active = this.getAttribute('data-active')==='1';
                        ['correct','wrong','doubt'].forEach(function(t){{
                            var b=document.getElementById('btn_'+t+'_{i}');
                            b.setAttribute('data-active','0');
                            b.style.background='#13132A';
                            b.style.borderColor='#555';
                            b.style.color='#888';
                        }});
                        var seg=document.getElementById('seg_{i}');
                        if(!active){{
                            this.setAttribute('data-active','1');
                            this.style.background='#0d2010';
                            this.style.borderColor='#22C55E';
                            this.style.color='#22C55E';
                            seg.style.borderLeftColor='#22C55E';
                            seg.style.background='#0d1f0d';
                            window._evals[{i}] = 'correct';
                        }} else {{
                            seg.style.borderLeftColor='#7c3aed';
                            seg.style.background='#1e1e2e';
                            delete window._evals[{i}];
                        }}
                    }}).call(this); return false;"
                    data-active="0"
                    style="width:36px;height:36px;border-radius:8px;
                           border:2px solid #555;background:#13132A;
                           color:#888;font-size:1.1rem;font-weight:bold;
                           cursor:pointer;">✓</button>

                <button id="btn_wrong_{i}"
                    onmousedown="(function(){{
                        window._evals = window._evals || {{}};
                        var active = this.getAttribute('data-active')==='1';
                        ['correct','wrong','doubt'].forEach(function(t){{
                            var b=document.getElementById('btn_'+t+'_{i}');
                            b.setAttribute('data-active','0');
                            b.style.background='#13132A';
                            b.style.borderColor='#555';
                            b.style.color='#888';
                        }});
                        var seg=document.getElementById('seg_{i}');
                        if(!active){{
                            this.setAttribute('data-active','1');
                            this.style.background='#200d0d';
                            this.style.borderColor='#EF4444';
                            this.style.color='#EF4444';
                            seg.style.borderLeftColor='#EF4444';
                            seg.style.background='#1f0d0d';
                            window._evals[{i}] = 'wrong';
                        }} else {{
                            seg.style.borderLeftColor='#7c3aed';
                            seg.style.background='#1e1e2e';
                            delete window._evals[{i}];
                        }}
                    }}).call(this); return false;"
                    data-active="0"
                    style="width:36px;height:36px;border-radius:8px;
                           border:2px solid #555;background:#13132A;
                           color:#888;font-size:1.1rem;font-weight:bold;
                           cursor:pointer;">✗</button>

                <button id="btn_doubt_{i}"
                    onmousedown="(function(){{
                        window._evals = window._evals || {{}};
                        var active = this.getAttribute('data-active')==='1';
                        ['correct','wrong','doubt'].forEach(function(t){{
                            var b=document.getElementById('btn_'+t+'_{i}');
                            b.setAttribute('data-active','0');
                            b.style.background='#13132A';
                            b.style.borderColor='#555';
                            b.style.color='#888';
                        }});
                        var seg=document.getElementById('seg_{i}');
                        if(!active){{
                            this.setAttribute('data-active','1');
                            this.style.background='#201a0d';
                            this.style.borderColor='#FACC15';
                            this.style.color='#FACC15';
                            seg.style.borderLeftColor='#FACC15';
                            seg.style.background='#1f1a0d';
                            window._evals[{i}] = 'doubt';
                        }} else {{
                            seg.style.borderLeftColor='#7c3aed';
                            seg.style.background='#1e1e2e';
                            delete window._evals[{i}];
                        }}
                    }}).call(this); return false;"
                    data-active="0"
                    style="width:36px;height:36px;border-radius:8px;
                           border:2px solid #555;background:#13132A;
                           color:#888;font-size:1.1rem;font-weight:bold;
                           cursor:pointer;">?</button>

            </div>
        </div>
        ''')

    hint_bar = f'''
    <script>
        // Initialize _evals once — don't overwrite if already set
        if (!window._evals) {{ window._evals = {{}}; }}
    </script>
    <div style="padding:8px 14px;background:#0d0d1a;border-radius:8px;
                margin-bottom:10px;font-size:0.78em;color:#555;
                border:1px solid #1e1e2e;">
        📋 {total_segments} segments &nbsp;·&nbsp;
        Mark each segment with ✓ ✗ ? then click
        <span style="color:#818cf8;">Evaluate Accuracy</span> below
    </div>
    '''

    return hint_bar + "\n".join(lines)

In [ ]:
# ══════════════════════════════════════════════════════════════
# VIDEO SUMMARISER ACCURACY — ROUGE Evaluation
# ══════════════════════════════════════════════════════════════
!pip install -q rouge-score
from rouge_score import rouge_scorer

def evaluate_summary_video_accuracy(full_segments, summary_timestamps, clip_duration=3.0):
    if not full_segments or not summary_timestamps:
        return None

    covered_windows = [
        (max(0, ts - 0.3), ts - 0.3 + clip_duration)
        for ts in summary_timestamps
    ]

    covered_texts, uncovered_texts = [], []
    for seg in full_segments:
        seg_time = seg["start"]
        in_clip  = any(start <= seg_time <= end for start, end in covered_windows)
        if in_clip:
            covered_texts.append(seg["text"])
        else:
            uncovered_texts.append(seg["text"])

    full_text    = " ".join(s["text"] for s in full_segments)
    covered_text = " ".join(covered_texts)

    if not covered_text.strip():
        print("⚠️ No transcript segments fall inside summary clips")
        return None

    coverage_pct = round(len(covered_texts) / len(full_segments) * 100, 1)
    scorer       = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores       = scorer.score(full_text, covered_text)

    r1 = round(scores['rouge1'].fmeasure * 100, 1)
    r2 = round(scores['rouge2'].fmeasure * 100, 1)
    rL = round(scores['rougeL'].fmeasure * 100, 1)

    avg = (r1 + r2 + rL) / 3
    if avg >= 60:   grade, color = "🟢 EXCELLENT", "#22C55E"
    elif avg >= 40: grade, color = "🟡 GOOD",      "#FACC15"
    elif avg >= 25: grade, color = "🟠 FAIR",      "#F97316"
    else:           grade, color = "🔴 POOR",      "#EF4444"

    print(f"\n{'='*54}")
    print(f"  {'SUMMARY VIDEO ACCURACY REPORT':^52}")
    print(f"{'='*54}")
    print(f"  Total segments     : {len(full_segments)}")
    print(f"  Segments in clips  : {len(covered_texts)}")
    print(f"  Segments skipped   : {len(uncovered_texts)}")
    print(f"  Coverage           : {coverage_pct}%")
    print(f"  ROUGE-1            : {r1}%")
    print(f"  ROUGE-2            : {r2}%")
    print(f"  ROUGE-L            : {rL}%")
    print(f"  Grade              : {grade}")
    print(f"{'='*54}")

    return {
        "coverage_pct": coverage_pct,
        "rouge1": r1, "rouge2": r2, "rougeL": rL,
        "grade": grade, "color": color,
        "covered_count": len(covered_texts),
        "total_count": len(full_segments),
        "covered_text": covered_text,
        "missed_texts": uncovered_texts
    }

print("✅ evaluate_summary_video_accuracy() ready")

  Preparing metadata (setup.py) ... done
✅ evaluate_summary_video_accuracy() ready


In [ ]:
# Pre-warm: make sure /content/ is writable and Gradio trusts it
import os, shutil, gradio as gr

test_path = "/content/summary_video.mp4"

# Create a dummy 0-byte file so Gradio sees the path exists at startup
if not os.path.exists(test_path):
    open(test_path, 'w').close()
    print(f"✅ Pre-created placeholder at {test_path}")
else:
    print(f"✅ Path already exists: {test_path}")

# Confirm Gradio version — v3 vs v4 behave differently
print(f"Gradio version: {gr.__version__}")

✅ Pre-created placeholder at /content/summary_video.mp4
Gradio version: 5.50.0


In [ ]:
import gradio as gr
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import gradio as gr
# ... other imports

_current_segments          = []
_current_summary           = ""
_current_chapters          = ""
_current_key_moments       = ""
_current_combined_summary  = ""
_current_frame_descs       = []
_current_visual_narrative  = ""
_current_video_path        = ""
_last_summary_timestamps   = []   # ← NEW
_last_clip_duration        = 3.0  # ← NEW
_semantic_major_texts      = []          # ← major nodes from semantic net
_semantic_useful_leaf_texts = []          # ← useful leaf nodes from semantic net

def run_pipeline(video_path):
    global _current_segments, _current_summary, _current_chapters
    global _current_key_moments, _current_combined_summary, _current_frame_descs, _current_visual_narrative

    if video_path is None:
        empty = go.Figure()
        return ("No video.", "—", "", "", "", "", "", "", "", [], "", empty, "")

    global _current_video_path
    _current_video_path = video_path

    # ── STEP 1: Detect media type ──────────────────────────────
    media_info = detect_media_type(video_path)
    print(media_info["message"])

    segments = []
    visual_narrative = ""
    frame_descs = []

    # ── STEP 2: Audio pipeline (only if audio exists) ──────────
    if media_info["has_audio"]:
        audio_path = extract_audio(video_path)
        segments   = transcribe_audio(audio_path)
        build_rag_index(segments)
        detected_lang = detect_language(segments)
        # ── Place name detection ──────────────────────────────
        _detected_places = extract_place_names(segments)
    else:
        detected_lang = "N/A — No audio track"
        _detected_places = []

    _current_segments = segments

    # Build transcript HTML
    lines = []
    for s in segments:
        lines.append(
            f'<div style="margin:4px 0;padding:8px 12px;border-left:4px solid #7c3aed;'
            f'background:#1e1e2e;border-radius:6px;font-family:monospace;">'
            f'<span style="color:#888;font-size:0.8em">[{s["start"]}s]</span> '
            f'<span style="color:#7c3aed;font-weight:600">{s["speaker"]}</span>: '
            f'<span style="color:#ddd">{s["text"]}</span></div>'
        )
    transcript_html = build_interactive_transcript_html(segments) if lines else "<p style='color:#888'>No audio track — transcript not available</p>"
    # ── STEP 3b: Visual pipeline runs before summary so we can pass it ──
    # (Visual pipeline block moved; we use visual_narrative computed below)
    # ── Build summary combining audio + visual ─────────────────────
    # We need to compute visual first if video exists
    _tmp_visual_narrative = ""
    if media_info["has_video"]:
        frames_tmp, timestamps_tmp = extract_keyframes(video_path)
        frame_descs_tmp = []
        for frame, ts in zip(frames_tmp, timestamps_tmp):
            desc = describe_frame(frame)
            desc["timestamp"] = ts
            frame_descs_tmp.append(desc)
        enriched_tmp = sync_frames_to_transcript(frame_descs_tmp, segments)
        _tmp_visual_narrative = generate_visual_narrative(enriched_tmp)
        # Store for later use
        frame_descs  = frame_descs_tmp
        frames       = frames_tmp
        synced_visual_html = build_synced_visual_html(enriched_tmp)
        visual_narrative   = _tmp_visual_narrative
    else:
        frames = []; frame_descs = []; synced_visual_html = "<p style='color:#888'>No video track — visual analysis not available</p>"
        visual_narrative = ""

    _current_frame_descs      = frame_descs
    _current_visual_narrative = visual_narrative

    if segments:
        _current_summary     = get_summary(segments, visual_narrative)
        _current_chapters    = get_chapters(segments, visual_narrative)
        _current_key_moments = get_key_moments(segments)
    elif visual_narrative:
        _current_summary     = visual_narrative
        _current_chapters    = "No audio track — chapters based on visual only."
        _current_key_moments = "No audio track detected."
    else:
        _current_summary     = "No audio track detected."
        _current_chapters    = "No audio track detected."
        _current_key_moments = "No audio track detected."

    # ── STEP 3: Visual pipeline handled above with summary ──────────

    # ── STEP 4: Combined summary ───────────────────────────────
    if _current_summary and visual_narrative:
        _current_combined_summary = get_combined_summary(_current_summary, visual_narrative)
    elif _current_summary:
        _current_combined_summary = _current_summary
    elif visual_narrative:
        _current_combined_summary = visual_narrative
    else:
        _current_combined_summary = "No content available."
    # Note: get_combined_summary is already updated to use movie-script style

    # ── STEP 5: Sentiment ──────────────────────────────────────
    sentiment_data = build_multimodal_sentiment_timeline(segments, frame_descs)

    # ── Place map HTML ────────────────────────────────────
    if _detected_places:
        query = "+".join(_detected_places[0].replace(" ", "+").split())
        all_places = ", ".join(_detected_places)
        place_map_html = f"""
        <div style="margin-top:14px;">
          <details open>
            <summary style="cursor:pointer;color:#7c3aed;font-weight:600;font-size:0.95rem;
                            padding:8px 12px;background:#1e1e2e;border-radius:8px;">
              🗺️ Places mentioned: {all_places} — click to collapse
            </summary>
            <div style="margin-top:8px;border-radius:10px;overflow:hidden;">
              <iframe
                src="https://maps.google.com/maps?q={query}&output=embed"
                width="100%" height="360"
                style="border:none;border-radius:10px;"
                allowfullscreen loading="lazy">
              </iframe>
            </div>
          </details>
        </div>
        """
        place_map_update = gr.update(value=place_map_html, visible=True)
    else:
        place_map_update = gr.update(value="", visible=False)

    return (
        transcript_html,
        f"🌐 Detected Language: **{detected_lang}** | 📁 Media: **{media_info['message']}**",
        _current_summary,
        _current_chapters,
        _current_key_moments,
        visual_narrative,
        synced_visual_html,
        _current_combined_summary,
        frames,
        build_multimodal_sentiment_figure(sentiment_data),
        _sentiment_stats_html(sentiment_data),
        gr.update(visible=True),
        place_map_update,
    )

def run_evaluation_auto():
    if not _current_segments:
        return "⚠️ Process a video first.", "", "", "", ""
    if not _current_combined_summary:
        return "⚠️ Run Analyze Video first.", "", "", "", ""
    # Retrieve visual narrative and audio summary from pipeline globals
    _audio_sum = _current_summary if _current_summary else None
    _vis_narr  = _current_visual_narrative if _current_visual_narrative else _current_combined_summary
    audio_pct, visual_pct, combined_pct, top_matches, explanation, audio_ref, visual_ref = evaluate_similarity_auto(
        _current_segments, _current_combined_summary,
        audio_summary=_audio_sum,
        visual_narrative=_vis_narr,
    )
    score_str = (
        f"🎯 Audio: {audio_pct}%  |  Visual: {visual_pct}%  |  Combined: {combined_pct}%"
        f"  (Auto | LLaMA 3.3)"
    )
    matches_str = "\n\n".join(
        f"[{round(t)}s]  score: {round(s*100,1)}%\n{text}"
        for s, text, t in top_matches
    )
    return score_str, matches_str, explanation, audio_ref, visual_ref

def run_evaluation_manual(user_reference):
    if not _current_segments:
        return "⚠️ Process a video first.", "", ""
    if not _current_combined_summary:
        return "⚠️ Run Analyze Video first.", "", ""
    if not user_reference.strip():
        return "⚠️ Please enter a reference text.", "", ""
    percentage, top_matches, explanation = evaluate_similarity_manual(
        _current_segments, _current_combined_summary, user_reference.strip()
    )
    score_str   = f"🎯 {percentage}% Semantic Match  (Manual | LLaMA 3.3)"
    matches_str = "\n\n".join(
        f"[{round(t)}s]  score: {round(s*100,1)}%\n{text}"
        for s, text, t in top_matches
    )
    return score_str, matches_str, explanation

def run_translation(target_lang):
    if not _current_segments:
        return "⚠️ Process a video first.", "", "", ""
    translated_segs = translate_transcript(_current_segments, target_lang)
    lines = []
    for s in translated_segs:
        lines.append(
            f'<div style="margin:4px 0;padding:8px 12px;border-left:4px solid #7c3aed;'
            f'background:#1e1e2e;border-radius:6px;font-family:monospace;">'
            f'<span style="color:#888;font-size:0.8em">[{s["start"]}s]</span> '
            f'<span style="color:#ddd">{s["text"]}</span></div>'
        )
    return (
        "\n".join(lines),
        translate_text_block(_current_summary,     target_lang),
        translate_text_block(_current_chapters,    target_lang),
        translate_text_block(_current_key_moments, target_lang)
    )

def chat_respond(message, history):
    answer  = answer_question(message)
    history = history or []
    history.append((message, answer))
    return "", history

def run_evaluation(reference_text):
    if not _current_segments:
        return "⚠️ Process a video first.", "", ""
    if not reference_text.strip():
        return "⚠️ Enter reference text.", "", ""
    if not _current_combined_summary:
        return "⚠️ Re-analyze the video first.", "", ""
    percentage, top_matches, explanation = evaluate_similarity(
        _current_segments, reference_text, _current_combined_summary
    )
    score_str = f"🎯 {percentage}% Semantic Match (Audio + Visual)"
    matches_str = "\n\n".join(
        f"[{round(t)}s] (score: {round(s*100,1)}%)\n{text}"
        for s, text, t in top_matches
    )
    return score_str, matches_str, explanation

CUSTOM_CSS = "body { background:#0f0f1a !important; } .gradio-container { max-width:1200px !important; } .tab-nav button { font-weight:600; font-size:0.95rem; }"

def run_summary_video(motion_thresh, clip_dur, max_frames):
    global _current_video_path, _last_summary_timestamps, _last_clip_duration
    if not _current_video_path:
        return None, "<p style='color:#EF4444'>⚠️ Please process a video first.</p>"
    try:
        kf, ts = extract_keyframes_motion(
            _current_video_path,
            max_frames=int(max_frames),
            motion_threshold=motion_thresh
        )
        if not ts:
            return None, "<p style='color:#EF4444'>⚠️ No motion detected — lower sensitivity.</p>"

        _last_summary_timestamps = ts
        _last_clip_duration      = clip_dur

        out_path = "/content/summary_video.mp4"
        result   = create_summary_video(
            _current_video_path, ts,
            output_path=out_path,
            clip_duration=clip_dur
        )

        if result and os.path.exists(result) and os.path.getsize(result) > 1000:
            size_kb = os.path.getsize(result) // 1024
            import base64
            with open(result, "rb") as f:
                video_b64 = base64.b64encode(f.read()).decode("utf-8")
            video_html = f"""
            <div style="background:#000;border-radius:10px;overflow:hidden;margin-top:8px;">
                <video controls autoplay style="width:100%;max-height:480px;">
                    <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
                </video>
            </div>
            <p style="color:#22C55E;margin-top:8px;font-size:0.9rem;">
                ✅ Done — {len(ts)} clips stitched ({size_kb} KB) |
                Total: {round(len(ts) * clip_dur)}s
            </p>
            """
            return result, video_html

        return None, "<p style='color:#EF4444'>❌ Video file empty or missing.</p>"

    except Exception as e:
        import traceback
        return None, f"<p style='color:#EF4444'>❌ {traceback.format_exc()}</p>"


_show_accuracy = {"visible": True}  # Global toggle state

_last_accuracy_result = None
_last_accuracy_missed_html = ""
_show_accuracy = {"visible": False}  # Hidden by default

def _render_accuracy_html(result, missed_html):
    return f"""
    <div style="background:rgba(108,99,255,0.08);border-radius:10px;padding:16px;margin-top:8px;">
      <div style="display:flex;gap:12px;margin-bottom:14px;">
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;padding:12px;
                    text-align:center;border-top:3px solid #6C63FF;">
          <div style="font-size:1.8rem;font-weight:700;color:#6C63FF;">{result['coverage_pct']}%</div>
          <div style="color:#aaa;font-size:0.82rem;">Segment Coverage</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;padding:12px;
                    text-align:center;border-top:3px solid #22C55E;">
          <div style="font-size:1.8rem;font-weight:700;color:#22C55E;">{result['rouge1']}%</div>
          <div style="color:#aaa;font-size:0.82rem;">ROUGE-1</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;padding:12px;
                    text-align:center;border-top:3px solid #818CF8;">
          <div style="font-size:1.8rem;font-weight:700;color:#818CF8;">{result['rouge2']}%</div>
          <div style="color:#aaa;font-size:0.82rem;">ROUGE-2</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;padding:12px;
                    text-align:center;border-top:3px solid #F59E0B;">
          <div style="font-size:1.8rem;font-weight:700;color:#F59E0B;">{result['rougeL']}%</div>
          <div style="color:#aaa;font-size:0.82rem;">ROUGE-L</div>
        </div>
      </div>
      <div style="padding:10px 14px;background:rgba(0,0,0,0.3);border-radius:8px;margin-bottom:12px;">
        <span style="color:{result['color']};font-size:1.1rem;font-weight:700;">{result['grade']}</span>
        <span style="color:#888;font-size:0.85rem;margin-left:12px;">
          {result['covered_count']} of {result['total_count']} transcript segments covered
        </span>
      </div>
      <div style="color:#aaa;font-size:0.85rem;margin-bottom:6px;">⚠️ Content not in summary:</div>
      {missed_html}
    </div>
    """

def run_accuracy_eval():
    global _last_accuracy_result, _last_accuracy_missed_html, _show_accuracy

    if not _current_segments:
        return (
            "<p style='color:#EF4444'>⚠️ Process a video first.</p>",
            gr.update(visible=False),
            gr.update(value="👁️ Show Report", visible=False)
        )
    if not _last_summary_timestamps:
        return (
            "<p style='color:#EF4444'>⚠️ Generate a summary video first.</p>",
            gr.update(visible=False),
            gr.update(value="👁️ Show Report", visible=False)
        )

    result = evaluate_summary_video_accuracy(
        _current_segments,
        _last_summary_timestamps,
        clip_duration=_last_clip_duration
    )
    if not result:
        return (
            "<p style='color:#EF4444'>❌ Could not evaluate.</p>",
            gr.update(visible=False),
            gr.update(value="👁️ Show Report", visible=False)
        )

    missed_html = "".join(
        f'<div style="margin:3px 0;padding:6px 10px;background:#1a0a0a;'
        f'border-left:3px solid #EF4444;border-radius:4px;color:#ccc;'
        f'font-size:0.82rem;">{t[:100]}</div>'
        for t in result["missed_texts"][:5]
    ) or "<p style='color:#555;font-size:0.85rem;'>None — full coverage!</p>"

    _last_accuracy_result = result
    _last_accuracy_missed_html = missed_html
    _show_accuracy["visible"] = False  # Always start hidden after fresh eval

    done_html = """
    <div style="padding:10px 16px;background:rgba(34,197,94,0.08);border-radius:8px;
                border-left:3px solid #22C55E;color:#22C55E;font-size:0.9rem;">
        ✅ Evaluation complete — click <b>Show Report</b> to view results.
    </div>
    """
    return (
        done_html,
        gr.update(value="", visible=False),   # report panel hidden
        gr.update(value="👁️ Show Report", visible=True)  # toggle button appears
    )


def toggle_accuracy_report(current_label):
    global _show_accuracy

    if not _last_accuracy_result:
        return (
            gr.update(value="<p style='color:#aaa'>Run evaluation first.</p>", visible=True),
            gr.update(value="👁️ Show Report")
        )

    _show_accuracy["visible"] = not _show_accuracy["visible"]

    if _show_accuracy["visible"]:
        return (
            gr.update(value=_render_accuracy_html(_last_accuracy_result, _last_accuracy_missed_html), visible=True),
            gr.update(value="🙈 Hide Report")
        )
    else:
        return (
            gr.update(value="", visible=False),
            gr.update(value="👁️ Show Report")
        )

def evaluate_transcript_accuracy(correct, wrong, doubt):
    """Called from JS with counts already computed in the browser."""
    total  = correct + wrong + doubt
    if total == 0:
        return "<p style='color:#888;padding:8px;'>No segments marked yet — use ✓ ✗ ? buttons first.</p>"

    accuracy   = round((correct / total) * 100, 1)
    unmarked   = len(_current_segments) - total if _current_segments else 0

    if accuracy >= 80:   grade, color = "🟢 EXCELLENT", "#22C55E"
    elif accuracy >= 60: grade, color = "🟡 GOOD",      "#FACC15"
    elif accuracy >= 40: grade, color = "🟠 FAIR",      "#F97316"
    else:                grade, color = "🔴 POOR",      "#EF4444"

    return f"""
    <div style="background:rgba(108,99,255,0.08);border-radius:10px;
                padding:16px;margin-top:8px;border:1px solid rgba(108,99,255,0.2);">
      <div style="display:flex;gap:12px;margin-bottom:14px;">
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                    padding:12px;text-align:center;border-top:3px solid {color};">
          <div style="font-size:2rem;font-weight:700;color:{color};">{accuracy}%</div>
          <div style="color:#aaa;font-size:0.82rem;">Transcript Accuracy</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                    padding:12px;text-align:center;border-top:3px solid #22C55E;">
          <div style="font-size:2rem;font-weight:700;color:#22C55E;">{correct}</div>
          <div style="color:#aaa;font-size:0.82rem;">✓ Correct</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                    padding:12px;text-align:center;border-top:3px solid #EF4444;">
          <div style="font-size:2rem;font-weight:700;color:#EF4444;">{wrong}</div>
          <div style="color:#aaa;font-size:0.82rem;">✗ Wrong</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                    padding:12px;text-align:center;border-top:3px solid #FACC15;">
          <div style="font-size:2rem;font-weight:700;color:#FACC15;">{doubt}</div>
          <div style="color:#aaa;font-size:0.82rem;">? Doubt</div>
        </div>
        <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                    padding:12px;text-align:center;border-top:3px solid #555;">
          <div style="font-size:2rem;font-weight:700;color:#888;">{unmarked}</div>
          <div style="color:#aaa;font-size:0.82rem;">Unmarked</div>
        </div>
      </div>
      <div style="padding:10px 14px;background:rgba(0,0,0,0.3);border-radius:8px;">
        <span style="color:{color};font-size:1.1rem;font-weight:700;">{grade}</span>
        <span style="color:#888;font-size:0.85rem;margin-left:12px;">
          Formula: ✓ ticked ÷ total marked × 100 = {correct} ÷ {total} × 100
        </span>
      </div>
    </div>
    """

with gr.Blocks(css=CUSTOM_CSS, title="AuraFlow") as demo:
    correct_state = gr.State(0)
    wrong_state   = gr.State(0)
    doubt_state   = gr.State(0)

    gr.HTML("""
    <div style="text-align:center;padding:24px 0 8px;">
      <h1 style="font-size:2.5rem;font-weight:800;margin:0;
                 background:linear-gradient(135deg,#7c3aed,#4f46e5,#06b6d4);
                 -webkit-background-clip:text;-webkit-text-fill-color:transparent;">
        AuraFlow
      </h1>

    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            video_input  = gr.Video(label="📁 Upload Video", height=280)
            run_btn      = gr.Button("🚀 Analyze Video", variant="primary", size="lg")
            lang_display = gr.Markdown("🌐 Detected Language: —")

    with gr.Tabs():

        with gr.Tab("📝 Transcript"):
          transcript_out = gr.HTML()
          eval_accuracy_btn = gr.Button(
            "Evaluate Accuracy",
            variant="primary",
            size="lg",
            visible=False
          )
          transcript_accuracy_out = gr.HTML(value="<div id='transcript_accuracy_box'></div>")
          place_map_out = gr.HTML(visible=False)

        # Add this inside your gr.Tabs() block


        with gr.Tab("📊 Analysis"):
            with gr.Row():
                summary_out  = gr.Textbox(label="📋 Summary",  lines=6,  interactive=False)
                chapters_out = gr.Textbox(label="📑 Chapters", lines=10, interactive=False)
            key_moments_out = gr.Textbox(label="⭐ Key Moments", lines=8, interactive=False)

        with gr.Tab("🎨 Visual"):
            visual_out        = gr.Textbox(label="🎬 Visual Narrative", lines=6, interactive=False)
            gr.HTML('<p style="color:#aaa;margin:8px 0 4px;">🔗 <b>Audio-Visual Sync Timeline</b></p>')
            synced_visual_out = gr.HTML()
            gallery_out       = gr.Gallery(label="🖼️ Keyframes", columns=5, height=200)

        with gr.Tab("📋 Combined Summary"):
            gr.HTML('<p style="color:#aaa;padding:4px 0 12px;">Combined audio + visual overview.</p>')
            combined_out = gr.Textbox(label="📋 Video Overview", lines=10, interactive=False)

        with gr.Tab("🎭 Multimodal Sentiment"):
            gr.HTML("""
            <div style="padding:4px 0 12px;">
              <p style="color:#aaa;margin:0 0 6px;">
                Emotional tone across the video — fusing <b style="color:#818CF8;">transcript text</b>
                + <b style="color:#F59E0B;">visual frames</b> into one
                <b style="color:#6C63FF;">combined score</b>.
              </p>
              <p style="color:#555;font-size:0.85rem;">
                — — — Text only &nbsp;|&nbsp; ········ Visual only &nbsp;|&nbsp;
                <span style="color:#6C63FF;">——</span> Fused (final)
              </p>
            </div>
            """)
            sentiment_stats_out = gr.HTML()
            sentiment_plot_out  = gr.Plot()

        with gr.Tab("🌍 Translation"):
            lang_selector    = gr.Dropdown(choices=list(LANGUAGES.keys()), value="Hindi", label="Language")
            translate_btn    = gr.Button("🔄 Translate", variant="secondary")
            t_transcript_out = gr.HTML()
            with gr.Row():
                t_summary_out  = gr.Textbox(label="Translated Summary",  lines=5, interactive=False)
                t_chapters_out = gr.Textbox(label="Translated Chapters", lines=8, interactive=False)
            t_moments_out = gr.Textbox(label="Translated Key Moments", lines=6, interactive=False)

        with gr.Tab("💬 Q&A Chat"):
            gr.HTML('<p style="color:#9CA3AF;padding:4px 0 8px;">Ask anything about the video — RAG + LLaMA</p>')
            chatbot = gr.Chatbot(height=420, type="messages")
            with gr.Row():
                chat_input = gr.Textbox(
                    placeholder='"What was the main argument?" or "What did the speaker say about AI?"',
                    label="Your question", scale=4
                )
                chat_btn = gr.Button("Ask", variant="primary", scale=1)

        with gr.Tab("🎯 Similarity Evaluation"):
            gr.HTML(
                '<p style="color:#aaa;padding:4px 0 8px;">' +
                '<b>Auto</b>: reference generated from audio transcript via LLaMA 3.3, ' +
                'scored against combined summary.<br>' +
                '<b>Manual</b>: paste your own reference text for comparison.' +
                '</p>'
            )
            auto_eval_btn         = gr.Button("🔍 Run Auto Evaluation  (LLaMA 3.3)", variant="primary")
            auto_score_out        = gr.Textbox(label="🎯 Similarity Scores — Audio | Visual | Combined", interactive=False)
            with gr.Row():
                auto_audio_ref_out  = gr.Textbox(label="📄 Auto-Generated Reference — Audio (via LLaMA 3.3)", lines=5, interactive=False)
                auto_visual_ref_out = gr.Textbox(label="📄 Auto-Generated Reference — Visual (via LLaMA 3.3)", lines=5, interactive=False)
            auto_matches_out      = gr.Textbox(label="🔗 Most Relevant Transcript Segments", lines=7, interactive=False)
            auto_explanation_out  = gr.Textbox(label="🧠 LLaMA 3.3 Analysis", lines=5, interactive=False)
            gr.HTML('<hr style="border:none;border-top:1px solid rgba(255,255,255,0.08);margin:12px 0;">')
            manual_ref_input       = gr.Textbox(
                label="📄 Your Reference Text",
                placeholder="Paste your own reference or expected content here...",
                lines=5
            )
            manual_eval_btn        = gr.Button("🔍 Run Manual Evaluation  (LLaMA 3.3)", variant="secondary")
            manual_score_out       = gr.Textbox(label="🎯 Similarity Score", interactive=False)
            manual_matches_out     = gr.Textbox(label="🔗 Most Relevant Transcript Segments", lines=7, interactive=False)
            manual_explanation_out = gr.Textbox(label="🧠 LLaMA 3.3 Analysis", lines=5, interactive=False)

        with gr.Tab("🎬 Summary Video"):
          gr.HTML('<p style="color:#aaa;padding:4px 0 8px;">Summarised video built from motion-detected keyframes.</p>')

          with gr.Row():
              motion_threshold = gr.Slider(
                  minimum=0.5, maximum=5.0, value=2.5, step=0.25,
                  label="Motion Sensitivity — lower = more keyframes captured"
              )
              clip_duration = gr.Slider(
                  minimum=1.0, maximum=6.0, value=3.0, step=0.5,
                  label="Clip Duration (seconds per keyframe)"
              )
              max_frames_slider = gr.Slider(
                  minimum=3, maximum=20, value=5, step=1,
                  label="Max Keyframes — more = longer summary video"
              )

          summary_video_btn  = gr.Button("🎬 Generate Summary Video", variant="primary")
          summary_file_out   = gr.File(label="⬇️ Download Summary Video", visible=True)
          summary_player_out = gr.HTML(label="📹 Summary Video Player")

          gr.HTML('<hr style="border:none;border-top:1px solid rgba(255,255,255,0.08);margin:16px 0;">')
          gr.HTML('<p style="color:#aaa;font-size:0.9rem;">📊 <b>Summary Accuracy</b> — measures how well clips represent the full video</p>')

          with gr.Row():
              accuracy_btn        = gr.Button("📊 Evaluate Summary Accuracy", variant="secondary", scale=3)
              accuracy_toggle_btn = gr.Button("👁️ Show Report", variant="secondary", scale=1, visible=False)

          accuracy_status_out = gr.HTML()                          # shows "✅ done" or errors
          accuracy_out        = gr.HTML(visible=False)             # hidden report panel

        # Add this inside your gr.Tabs() block
        with gr.Tab("🕸️ Semantic Network"):
            gr.HTML('<p style="color:#aaa;padding:4px 0 8px;">Shows which parts of the transcript are semantically related to each other.</p>')
            semantic_threshold = gr.Slider(
                minimum=0.3, maximum=0.9, value=0.5, step=0.05,
                label="Similarity Threshold — higher = only show strongest connections"
            )
            semantic_btn         = gr.Button("🕸️ Build Semantic Network", variant="secondary")
            semantic_plot        = gr.Plot()
            semantic_summary_btn = gr.Button("🧠 Summarize from Semantic Net", variant="primary")
            semantic_summary_out = gr.Textbox(
                label="🧠 Major-Node Audio Summary",
                lines=5, interactive=False,
                placeholder="Build the network first, then click Summarize."
            )

        # Add the click function
        _semantic_major_texts = []

        def run_semantic_network(threshold):
            global _semantic_major_texts, _semantic_useful_leaf_texts
            if not _current_segments:
                return go.Figure()
            fig, major_texts, useful_leaf_texts = build_semantic_network(_current_segments, threshold=threshold)
            _semantic_major_texts = major_texts
            _semantic_useful_leaf_texts = useful_leaf_texts
            return fig

        def run_semantic_summary():
          if not _semantic_major_texts and not _semantic_useful_leaf_texts:
            return "⚠️ Build the semantic network first."
          major_block = " ".join(_semantic_major_texts)
          leaf_block  = " ".join(_semantic_useful_leaf_texts)
          prompt = (
            f"Major nodes (core topics):\n{major_block}\n\n"
            f"Supporting nodes (relevant details):\n{leaf_block}\n\n"
            f"Summarize both sections together in 4-5 sentences, "
            f"covering the main themes from the major nodes and the "
            f"additional context from the supporting nodes."
          )
          resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=350, temperature=0.1
          )
          return resp.choices[0].message.content.strip()

        semantic_btn.click(
            fn=run_semantic_network,
            inputs=[semantic_threshold],
            outputs=[semantic_plot]
        )
        semantic_summary_btn.click(
            fn=run_semantic_summary,
            inputs=[],
            outputs=[semantic_summary_out]
        )

    run_btn.click(
        fn=run_pipeline,
        inputs=[video_input],
        outputs=[
            transcript_out, lang_display,
            summary_out, chapters_out, key_moments_out,
            visual_out, synced_visual_out,
            combined_out, gallery_out,
            sentiment_plot_out, sentiment_stats_out,
            eval_accuracy_btn,
            place_map_out,
        ]
    )

    translate_btn.click(
        fn=run_translation,
        inputs=[lang_selector],
        outputs=[t_transcript_out, t_summary_out, t_chapters_out, t_moments_out]
    )

    chat_btn.click(fn=chat_respond, inputs=[chat_input, chatbot], outputs=[chat_input, chatbot])
    chat_input.submit(fn=chat_respond, inputs=[chat_input, chatbot], outputs=[chat_input, chatbot])

    auto_eval_btn.click(
        fn=run_evaluation_auto,
        inputs=[],
        outputs=[auto_score_out, auto_matches_out, auto_explanation_out, auto_audio_ref_out, auto_visual_ref_out]
    )
    manual_eval_btn.click(
        fn=run_evaluation_manual,
        inputs=[manual_ref_input],
        outputs=[manual_score_out, manual_matches_out, manual_explanation_out]
    )

    summary_video_btn.click(
        fn=run_summary_video,
        inputs=[motion_threshold, clip_duration, max_frames_slider],
        outputs=[summary_file_out, summary_player_out]
    )

    accuracy_btn.click(
    fn=run_accuracy_eval,
    inputs=[],
    outputs=[accuracy_status_out, accuracy_out, accuracy_toggle_btn]
    )

    accuracy_toggle_btn.click(
        fn=toggle_accuracy_report,
        inputs=[accuracy_toggle_btn],
        outputs=[accuracy_out, accuracy_toggle_btn]
    )
    eval_accuracy_btn.click(
      fn=None,   # ← No Python function at all
      inputs=[],
      outputs=[],
      js="""
      () => {
          // Collect from window._evals in THIS iframe context
          const evals = window._evals || {};
          let correct = 0, wrong = 0, doubt = 0;
          Object.values(evals).forEach(function(v) {
              if (v === 'correct') correct++;
              else if (v === 'wrong') wrong++;
              else if (v === 'doubt') doubt++;
          });

          const total = correct + wrong + doubt;

          // Find the transcript_accuracy_out HTML component and write into it
          // Gradio renders gr.HTML as a div — find it by walking the DOM
          const allDivs = document.querySelectorAll('.prose, [data-testid="html"]');
          let targetEl = null;
          allDivs.forEach(function(el) {
              if (el.id && el.id.includes('transcript_accuracy')) {
                  targetEl = el;
              }
          });

          // Fallback: find the div right after the Evaluate Accuracy button
          if (!targetEl) {
              const btns = document.querySelectorAll('button');
              btns.forEach(function(btn) {
                  if (btn.textContent.trim() === 'Evaluate Accuracy') {
                      // Walk up to find parent, then find next sibling HTML component
                      let parent = btn.closest('.block, .wrap, [class*="container"]');
                      if (parent && parent.nextElementSibling) {
                          targetEl = parent.nextElementSibling.querySelector('div');
                      }
                  }
              });
          }

          // Build result HTML
          let html = '';
          if (total === 0) {
              html = "<p style='color:#888;padding:12px;background:#1a1a2e;border-radius:8px;'>No segments marked yet — use ✓ ✗ ? buttons first.</p>";
          } else {
              const accuracy = ((correct / total) * 100).toFixed(1);
              let grade, color;
              if (accuracy >= 80)      { grade = '🟢 EXCELLENT'; color = '#22C55E'; }
              else if (accuracy >= 60) { grade = '🟡 GOOD';      color = '#FACC15'; }
              else if (accuracy >= 40) { grade = '🟠 FAIR';      color = '#F97316'; }
              else                     { grade = '🔴 POOR';      color = '#EF4444'; }

              html = `
              <div style="background:rgba(108,99,255,0.08);border-radius:10px;
                          padding:16px;margin-top:8px;border:1px solid rgba(108,99,255,0.2);">
                <div style="display:flex;gap:12px;margin-bottom:14px;">
                  <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                              padding:12px;text-align:center;border-top:3px solid ${color};">
                    <div style="font-size:2rem;font-weight:700;color:${color};">${accuracy}%</div>
                    <div style="color:#aaa;font-size:0.82rem;">Transcript Accuracy</div>
                  </div>
                  <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                              padding:12px;text-align:center;border-top:3px solid #22C55E;">
                    <div style="font-size:2rem;font-weight:700;color:#22C55E;">${correct}</div>
                    <div style="color:#aaa;font-size:0.82rem;">✓ Correct</div>
                  </div>
                  <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                              padding:12px;text-align:center;border-top:3px solid #EF4444;">
                    <div style="font-size:2rem;font-weight:700;color:#EF4444;">${wrong}</div>
                    <div style="color:#aaa;font-size:0.82rem;">✗ Wrong</div>
                  </div>
                  <div style="flex:1;background:rgba(255,255,255,0.05);border-radius:8px;
                              padding:12px;text-align:center;border-top:3px solid #FACC15;">
                    <div style="font-size:2rem;font-weight:700;color:#FACC15;">${doubt}</div>
                    <div style="color:#aaa;font-size:0.82rem;">? Doubt</div>
                  </div>
                </div>
                <div style="padding:10px 14px;background:rgba(0,0,0,0.3);border-radius:8px;">
                  <span style="font-size:1.1rem;font-weight:700;color:${color};">${grade}</span>
                </div>
              </div>`;
          }

          // Write to the output element — find any empty div below the button area
          // Most reliable: use the component that Gradio renders for transcript_accuracy_out
          const box = document.getElementById('transcript_accuracy_box');
          if (box) {
              box.innerHTML = html;
          }

          return [];
      }
      """
  )


import os, warnings
warnings.filterwarnings("ignore")

# Suppress the Gradio manifest 404 errors — these come from the browser
# trying to register the page as a PWA (Progressive Web App).
# They are harmless browser-only requests and do not affect functionality.
# The fix: add a minimal manifest.json to Gradio's static folder.
try:
    import gradio
    gradio_static = os.path.join(os.path.dirname(gradio.__file__), "templates", "frontend")
    manifest_path = os.path.join(gradio_static, "manifest.json")
    if not os.path.exists(manifest_path):
        import json as _json
        _manifest = {
            "name": "AuraFlow",
            "short_name": "AuraFlow",
            "start_url": ".",
            "display": "standalone",
            "background_color": "#0f0f1a",
            "theme_color": "#6C63FF",
            "icons": []
        }
        os.makedirs(gradio_static, exist_ok=True)
        with open(manifest_path, "w") as _f:
            _json.dump(_manifest, _f)
        print("✅ Gradio manifest.json created — 404 errors suppressed")
except Exception as _e:
    print(f"⚠️ Could not create manifest.json: {_e} (harmless)")

demo.launch(
    share=True,
    debug=False,
    allowed_paths=["/content/", "/tmp/"],
    server_name="0.0.0.0",
    quiet=True,          # suppresses repeated status messages
    show_error=True,
)

* Running on public URL: https://6b71eaf43f71662c71.gradio.live


In [ ]:
import torch, numpy as np

def _score_text_sentiment(text):
    enc = _sentiment_tokenizer(text, truncation=True, padding=True,
                               max_length=96, return_tensors='pt')
    enc = {k: v.to(_sentiment_model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = _sentiment_model(**enc).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    probs[1] *= 1.28
    probs[2] *= 1.12
    probs = probs / probs.sum()
    pred_id = int(np.argmax(probs))
    id2label = {0:'Negative', 1:'Neutral', 2:'Positive'}
    label = id2label[pred_id]
    display_score = round(float(max(0, min(1, 0.5 + (probs[2]-probs[0])/2))), 3)
    return {"label": label, "score": display_score, "confidence": round(float(probs[pred_id]),3)}

globals()['_score_text_sentiment'] = _score_text_sentiment
print("✅ Done — run accuracy test now")

✅ Done — run accuracy test now


In [ ]:
import torch, numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import plotly.figure_factory as ff

# ── Apply boost ────────────────────────────────────────
def _score_text_sentiment(text):
    enc = _sentiment_tokenizer(text, truncation=True, padding=True,
                               max_length=96, return_tensors='pt')
    enc = {k: v.to(_sentiment_model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = _sentiment_model(**enc).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    probs[1] *= 1.28
    probs[2] *= 1.12
    probs = probs / probs.sum()
    pred_id = int(np.argmax(probs))
    id2label = {0:'Negative', 1:'Neutral', 2:'Positive'}
    label = id2label[pred_id]
    display_score = round(float(max(0, min(1, 0.5 + (probs[2]-probs[0])/2))), 3)
    return {"label": label, "score": display_score, "confidence": round(float(probs[pred_id]),3)}

globals()['_score_text_sentiment'] = _score_text_sentiment

# ── Accuracy test ──────────────────────────────────────
label_normalize = {'Positive':'Positive','Negative':'Negative',
                   'Neutral':'Neutral','Irrelevant':'Neutral'}
df['sentiment_n'] = df['sentiment'].map(label_normalize)
df2 = df.dropna(subset=['sentiment_n','text'])
df2 = df2[df2['text'].str.len() > 3]

groups = []
for lbl, grp in df2.groupby('sentiment_n'):
    groups.append(grp.sample(100, random_state=99))
sample = pd.concat(groups).reset_index(drop=True)

print("Running on 300 tweets...")
preds = [_score_text_sentiment(t)["label"].capitalize()
         for t in sample['text'].tolist()]
true  = sample['sentiment_n'].tolist()

print("\n" + "="*54)
print(f"  {'AURAFLOW SENTIMENT ACCURACY REPORT':^52}")
print("="*54)
overall = round(sum(t==p for t,p in zip(true,preds))/len(true)*100,1)
all_pass = True
for cat in ['Positive','Negative','Neutral']:
    sub = [(t,p) for t,p in zip(true,preds) if t==cat]
    acc = round(sum(t==p for t,p in sub)/len(sub)*100,1)
    bar = "█"*int(acc/5) + "░"*(20-int(acc/5))
    status = "✅ PASS" if acc >= 85 else "❌ FAIL"
    if acc < 85: all_pass = False
    print(f"  {cat:10}  {bar}  {acc:5.1f}%  {status}")
print("-"*54)
print(f"  {'Overall':10}  {'█'*int(overall/5)}{'░'*(20-int(overall/5))}  {overall:5.1f}%")
print("="*54)
print(classification_report(true, preds,
      target_names=['Negative','Neutral','Positive']))
print("🎉 All classes 85%+!" if all_pass else "⚠️ Some classes below 85% — paste results here")

# ── Confusion matrix ───────────────────────────────────
cm = confusion_matrix(true, preds, labels=['Negative','Neutral','Positive'])
fig_cm = ff.create_annotated_heatmap(
    z=cm, x=['Pred Neg','Pred Neu','Pred Pos'],
    y=['Actual Neg','Actual Neu','Actual Pos'],
    colorscale='Purples', showscale=True,
    annotation_text=cm.astype(str)
)
fig_cm.update_layout(
    title=dict(text='Confusion Matrix — AuraFlow Sentiment',
               font=dict(size=15, color='#6C63FF')),
    paper_bgcolor='#0E1117', font=dict(color='#FAFAFA'),
    width=500, height=420, margin=dict(t=60,b=40,l=40,r=40)
)
fig_cm.show()

# ── Bar chart ──────────────────────────────────────────
results = {}
for cat in ['Positive','Negative','Neutral']:
    sub = [(t,p) for t,p in zip(true,preds) if t==cat]
    results[cat] = round(sum(t==p for t,p in sub)/len(sub)*100,1)

fig_bar = go.Figure(go.Bar(
    x=['Negative','Neutral','Positive','Overall'],
    y=[results['Negative'],results['Neutral'],results['Positive'],overall],
    marker_color=[
        '#22C55E' if results['Negative']>=85 else '#EF4444',
        '#22C55E' if results['Neutral'] >=85 else '#EF4444',
        '#22C55E' if results['Positive']>=85 else '#EF4444',
        '#6C63FF'
    ],
    text=[f"{results['Negative']}%",f"{results['Neutral']}%",
          f"{results['Positive']}%",f"{overall}%"],
    textposition='outside', textfont=dict(size=14, color='#FAFAFA'),
))
fig_bar.add_hline(y=85, line_dash="dash", line_color="#FACC15",
                  annotation_text="85% Target", annotation_font_color="#FACC15")
fig_bar.update_layout(
    title=dict(text='Per-Class Accuracy vs 85% Target',
               font=dict(size=15, color='#6C63FF')),
    paper_bgcolor='#0E1117', plot_bgcolor='#1E2130',
    font=dict(color='#FAFAFA'),
    yaxis=dict(range=[0,115], title='Accuracy %', gridcolor='#2E3347'),
    xaxis=dict(title='Sentiment Class'),
    width=500, height=380, margin=dict(t=60,b=40,l=40,r=40)
)
fig_bar.show()

Running on 300 tweets...

           AURAFLOW SENTIMENT ACCURACY REPORT         
  Positive    █████████████████░░░   85.0%  ✅ PASS
  Negative    ██████████████████░░   91.0%  ✅ PASS
  Neutral     █████████████████░░░   89.0%  ✅ PASS
------------------------------------------------------
  Overall     █████████████████░░░   88.3%
              precision    recall  f1-score   support

    Negative       0.89      0.91      0.90       100
     Neutral       0.83      0.89      0.86       100
    Positive       0.93      0.85      0.89       100

    accuracy                           0.88       300
   macro avg       0.89      0.88      0.88       300
weighted avg       0.89      0.88      0.88       300

🎉 All classes 85%+!
